## Run guide

This notebook records an exploratory Kaggle audit. Outputs were cleared for the revised export except for the saved six-model checkpoint inventory. The full notebook has not been rerun.

- **Verify inputs first.** Attach the project model code, tokenizer, and checkpoints at the paths referenced below. Pin the pretrained `mha-s1-ckpts` dataset to **version 1**. Search for `SIX-MODEL CHECKPOINT INVENTORY` and run that cell before loading any models. It should find `ckpts/mha_s1/final.pt` with 12 query heads and 12 KV heads. The saved inventory establishes the observed head counts. The previous export does not record the attached dataset version.
- **Recreate the summary and bar charts from saved results.** Place `six_model_concentration_summary.csv` in `/kaggle/working/`. Run the cells that save `components_to_80_comparison.png` and `top_component_damage_comparison.png`. To regenerate the heatmaps and cumulative plot, also supply the per-model `*_grouped_results.csv` and `*_cumulative_ablation.csv` files to the plotting cells.
- **Recompute ablations selectively.** Set up the IOI generator, fixed 500-prompt evaluation set, tokenizer, model module, and ablation helpers before running the grouped audit and cumulative analysis. Use a fresh output directory or verify any resumed CSVs came from the same checkpoint and prompt set.
- **Historical trials.** Original notebook cells 25, 26, 27, 40, 51, 52, 84, 95, 110, 118, 129, 133, 168, 182, 186 had saved errors or interruptions. They are tagged `historical-error` here. Skip these on a review pass. Any cell that uses a variable or file from a skipped trial requires that input to be restored or recomputed first. Original cell numbers exclude this guide.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import sys
import os

CODE_DIR = "/kaggle/input/datasets/nainikabatchu/archive"

# Add the Part 2 code folder to Python's import path
sys.path.append(CODE_DIR)

# Check the important files exist
print("Model exists:", os.path.exists(f"{CODE_DIR}/model.py"))
print("Checkpoint loader exists:", os.path.exists(f"{CODE_DIR}/checkpoint.py"))

# Try importing the model code
import model

print("\nSuccessfully imported model.py!")

In [ ]:
import os

# Show all files attached to the notebook
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if "archive" not in dirname:  # focus on the newly uploaded checkpoint dataset
            print(os.path.join(dirname, filename))

In [ ]:
import os

print("Contents of /kaggle/input:\n")

for dirname, _, filenames in os.walk("/kaggle/input"):
    print("Folder:", dirname)
    for filename in filenames:
        print("   File:", filename)

In [ ]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
CODE_DIR = "/kaggle/input/datasets/nainikabatchu/archive"

print("=== checkpoint.py ===\n")
with open(f"{CODE_DIR}/checkpoint.py", "r") as f:
    print(f.read())

print("\n=== model.py ===\n")
with open(f"{CODE_DIR}/model.py", "r") as f:
    print(f.read())

In [ ]:
import sys
import torch

# Add Part 2 code to Python's import path
CODE_DIR = "/kaggle/input/datasets/nainikabatchu/archive"
sys.path.append(CODE_DIR)

from checkpoint import load_model

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Path to a mid-training checkpoint
CKPT_PATH = "/kaggle/input/datasets/nainikabatchu/mha-s0-ckpts2/ckpts/mha_s0/step_000763.pt"

# Load model
model, info = load_model(CKPT_PATH, device=device)

print("\nModel loaded successfully!")
print("Checkpoint step:", info["step"])
print("Tokens seen:", info["tokens_seen"])
print("Model configuration:")
print(info["config"])

In [ ]:
# Check the model structure
print(model)

In [ ]:
# Create a small dummy input
x = torch.randint(
    low=0,
    high=info["config"]["vocab_size"],
    size=(1, 10),
    device=device
)

# Run the model
model.eval()

with torch.no_grad():
    output = model(x)

print("Input shape:", x.shape)

# Check output type and shape
print("Output type:", type(output))

if isinstance(output, tuple):
    print("Output shape:", output[0].shape)
else:
    print("Output shape:", output.shape)

In [ ]:
import os

print("=== ALL TOKENIZER FILES ===")
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if "tokenizer" in filename.lower():
            print(os.path.join(dirname, filename))

print("\n=== ALL CHECKPOINT FILES ===")
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".pt"):
            print(os.path.join(dirname, filename))

In [ ]:
!pip install -q tokenizers

In [ ]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/nainikabatchu/tokenizer-32000-json/tokenizer_32000.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.get_vocab_size())

In [ ]:
import random

NAMES = [
    "Alice", "Bob", "Charlie", "David",
    "Emma", "Grace", "Henry", "Jack",
    "Mary", "Sarah", "Tom", "William"
]

OBJECTS = [
    "book", "ball", "gift", "letter",
    "toy", "flower"
]

PLACES = [
    "park", "store", "school", "garden"
]


def generate_ioi_prompt():
    """Generate one synthetic IOI example."""

    name_a, name_b = random.sample(NAMES, 2)
    obj = random.choice(OBJECTS)
    place = random.choice(PLACES)

    # IOI-style prompt: the correct answer is name_b
    prompt = (
        f"{name_a} and {name_b} went to the {place}. "
        f"{name_a} gave the {obj} to {name_b}. "
        f"Then {name_a} gave another {obj} to"
    )

    return {
        "prompt": prompt,
        "correct": name_b,
        "incorrect": name_a
    }


# Generate and display one example
example = generate_ioi_prompt()

print("PROMPT:")
print(example["prompt"])
print("\nCorrect answer:", example["correct"])
print("Incorrect answer:", example["incorrect"])

In [ ]:
# Encode the prompt
prompt = example["prompt"]
encoded = tokenizer.encode(prompt)

input_ids = torch.tensor(
    [encoded.ids],
    dtype=torch.long,
    device=device
)

print("Prompt:")
print(prompt)

print("\nNumber of tokens:", len(encoded.ids))
print("Last few token IDs:", encoded.ids[-10:])

# Run the model
model.eval()

with torch.no_grad():
    logits, _ = model(input_ids)

print("\nLogits shape:", logits.shape)

In [ ]:
# Check how the answer names are tokenized

correct_text = " " + example["correct"]
incorrect_text = " " + example["incorrect"]

correct_encoding = tokenizer.encode(correct_text)
incorrect_encoding = tokenizer.encode(incorrect_text)

print("Correct answer:", repr(correct_text))
print("Token IDs:", correct_encoding.ids)
print("Tokens:", correct_encoding.tokens)

print("\nIncorrect answer:", repr(incorrect_text))
print("Token IDs:", incorrect_encoding.ids)
print("Tokens:", incorrect_encoding.tokens)

In [ ]:
# Get the logits for the NEXT token after the prompt
next_token_logits = logits[0, -1]

correct_id = correct_encoding.ids[0]
incorrect_id = incorrect_encoding.ids[0]

correct_score = next_token_logits[correct_id].item()
incorrect_score = next_token_logits[incorrect_id].item()

print("Prompt:")
print(prompt)

print("\nCorrect answer:", example["correct"])
print("Correct score:", round(correct_score, 4))

print("\nIncorrect answer:", example["incorrect"])
print("Incorrect score:", round(incorrect_score, 4))

if correct_score > incorrect_score:
    print("\nRESULT: Model prefers the CORRECT answer.")
else:
    print("\nRESULT: Model prefers the INCORRECT answer.")

In [ ]:
def get_single_token_id(text):
    """Return the token ID if text is exactly one token, otherwise None."""
    encoding = tokenizer.encode(" " + text)
    if len(encoding.ids) == 1:
        return encoding.ids[0]
    return None


def evaluate_ioi(model, n_examples=50):
    results = []

    model.eval()

    for _ in range(n_examples):
        ex = generate_ioi_prompt()

        correct_id = get_single_token_id(ex["correct"])
        incorrect_id = get_single_token_id(ex["incorrect"])

        # Skip examples where either name is split into multiple tokens
        if correct_id is None or incorrect_id is None:
            continue

        # Tokenize prompt
        ids = tokenizer.encode(ex["prompt"]).ids
        input_ids = torch.tensor(
            [ids],
            dtype=torch.long,
            device=device
        )

        # Run model
        with torch.no_grad():
            batch_logits, _ = model(input_ids)

        next_logits = batch_logits[0, -1]

        correct_score = next_logits[correct_id].item()
        incorrect_score = next_logits[incorrect_id].item()

        results.append({
            "correct": ex["correct"],
            "incorrect": ex["incorrect"],
            "correct_score": correct_score,
            "incorrect_score": incorrect_score,
            "success": correct_score > incorrect_score
        })

    accuracy = sum(r["success"] for r in results) / len(results)

    return accuracy, results


accuracy, results = evaluate_ioi(model, n_examples=50)

print("Valid examples tested:", len(results))
print("IOI pairwise accuracy:", f"{accuracy * 100:.1f}%")

In [ ]:
# Path to an earlier checkpoint
EARLY_CKPT_PATH = (
    "/kaggle/input/datasets/nainikabatchu/"
    "mha-s0-ckpts2/ckpts/mha_s0/step_000191.pt"
)

# Load the earlier model
early_model, early_info = load_model(EARLY_CKPT_PATH, device=device)

print("Early checkpoint loaded!")
print("Step:", early_info["step"])
print("Tokens seen:", early_info["tokens_seen"])

# Run the same IOI evaluation
early_accuracy, early_results = evaluate_ioi(
    early_model,
    n_examples=50
)

print("\nValid examples tested:", len(early_results))
print("IOI pairwise accuracy:", f"{early_accuracy * 100:.1f}%")

In [ ]:
# Create a fixed IOI evaluation set
random.seed(42)

FIXED_IOI_SET = []

while len(FIXED_IOI_SET) < 100:
    ex = generate_ioi_prompt()

    correct_id = get_single_token_id(ex["correct"])
    incorrect_id = get_single_token_id(ex["incorrect"])

    if correct_id is not None and incorrect_id is not None:
        FIXED_IOI_SET.append(ex)

print("Fixed IOI test set created!")
print("Number of examples:", len(FIXED_IOI_SET))

print("\nFirst example:")
print(FIXED_IOI_SET[0])

In [ ]:
def evaluate_fixed_ioi(model, test_set):
    results = []

    model.eval()

    for ex in test_set:
        correct_id = get_single_token_id(ex["correct"])
        incorrect_id = get_single_token_id(ex["incorrect"])

        ids = tokenizer.encode(ex["prompt"]).ids
        input_ids = torch.tensor(
            [ids],
            dtype=torch.long,
            device=device
        )

        with torch.no_grad():
            batch_logits, _ = model(input_ids)

        next_logits = batch_logits[0, -1]

        correct_score = next_logits[correct_id].item()
        incorrect_score = next_logits[incorrect_id].item()

        results.append({
            "success": correct_score > incorrect_score,
            "correct_score": correct_score,
            "incorrect_score": incorrect_score
        })

    accuracy = sum(r["success"] for r in results) / len(results)
    return accuracy, results

In [ ]:
# Evaluate both checkpoints on the SAME 100 examples

early_fixed_acc, _ = evaluate_fixed_ioi(
    early_model,
    FIXED_IOI_SET
)

mid_fixed_acc, _ = evaluate_fixed_ioi(
    model,
    FIXED_IOI_SET
)

print("=== FIXED IOI EVALUATION ===")
print(f"Step 191: {early_fixed_acc * 100:.1f}%")
print(f"Step 763: {mid_fixed_acc * 100:.1f}%")

In [ ]:
FINAL_CKPT_PATH = (
    "/kaggle/input/datasets/nainikabatchu/"
    "mha-s0-ckpts2/ckpts/mha_s0/final.pt"
)

# Load final model
final_model, final_info = load_model(FINAL_CKPT_PATH, device=device)

print("Final checkpoint loaded!")
print("Step:", final_info["step"])
print("Tokens seen:", final_info["tokens_seen"])

# Evaluate on the exact same 100 examples
final_fixed_acc, final_results = evaluate_fixed_ioi(
    final_model,
    FIXED_IOI_SET
)

print("\n=== FINAL IOI RESULT ===")
print(f"Final checkpoint accuracy: {final_fixed_acc * 100:.1f}%")

In [ ]:
def get_ioi_metrics(model, test_set):
    """Compute baseline IOI accuracy and logit margins."""
    
    results = []
    model.eval()

    for ex in test_set:
        correct_id = get_single_token_id(ex["correct"])
        incorrect_id = get_single_token_id(ex["incorrect"])

        ids = tokenizer.encode(ex["prompt"]).ids
        input_ids = torch.tensor(
            [ids],
            dtype=torch.long,
            device=device
        )

        with torch.no_grad():
            batch_logits, _ = model(input_ids)

        next_logits = batch_logits[0, -1]

        correct_score = next_logits[correct_id].item()
        incorrect_score = next_logits[incorrect_id].item()
        margin = correct_score - incorrect_score

        results.append({
            "success": margin > 0,
            "margin": margin
        })

    accuracy = sum(r["success"] for r in results) / len(results)
    mean_margin = sum(r["margin"] for r in results) / len(results)

    return accuracy, mean_margin, results


# Calculate the baseline for the final model
baseline_acc, baseline_margin, baseline_results = get_ioi_metrics(
    final_model,
    FIXED_IOI_SET
)

print("=== FINAL MODEL BASELINE ===")
print(f"Accuracy: {baseline_acc * 100:.1f}%")
print(f"Mean IOI logit margin: {baseline_margin:.4f}")

In [ ]:
from model import ablate_heads

# Test: ablate head 0 in layer 0
with ablate_heads(final_model, {0: [0]}):
    test_acc, test_margin, test_results = get_ioi_metrics(
        final_model,
        FIXED_IOI_SET
    )

print("=== SINGLE HEAD ABLATION TEST ===")
print(f"Baseline accuracy: {baseline_acc * 100:.1f}%")
print(f"Ablated accuracy:  {test_acc * 100:.1f}%")

print(f"\nBaseline margin: {baseline_margin:.4f}")
print(f"Ablated margin:  {test_margin:.4f}")
print(f"Margin drop:     {baseline_margin - test_margin:.4f}")

In [ ]:
import pandas as pd
import time

ablation_results = []

n_layers = final_model.cfg.n_layers
n_heads = final_model.cfg.n_heads

print(f"Testing {n_layers} layers × {n_heads} heads = {n_layers * n_heads} ablations")
print("This may take some time on CPU...")

start_time = time.time()

for layer in range(n_layers):
    for head in range(n_heads):

        # Ablate one head
        with ablate_heads(final_model, {layer: [head]}):
            acc, margin, _ = get_ioi_metrics(
                final_model,
                FIXED_IOI_SET
            )

        ablation_results.append({
            "layer": layer,
            "head": head,
            "accuracy": acc,
            "mean_margin": margin,
            "accuracy_drop_pp": (baseline_acc - acc) * 100,
            "margin_drop": baseline_margin - margin
        })

    print(f"Completed layer {layer}")

elapsed = time.time() - start_time

results_df = pd.DataFrame(ablation_results)

print(f"\nCompleted in {elapsed / 60:.1f} minutes")
print("\nTop 10 heads by margin drop:")
print(
    results_df
    .sort_values("margin_drop", ascending=False)
    .head(10)
    [["layer", "head", "accuracy", "mean_margin",
      "accuracy_drop_pp", "margin_drop"]]
)

In [ ]:
# Sort heads by their positive margin damage
sorted_results = results_df.sort_values(
    "margin_drop",
    ascending=False
).copy()

# Only count heads that actually hurt performance when removed
positive_damage = sorted_results[
    sorted_results["margin_drop"] > 0
].copy()

# Calculate cumulative damage
total_damage = positive_damage["margin_drop"].sum()

positive_damage["cumulative_damage"] = (
    positive_damage["margin_drop"].cumsum()
)

positive_damage["cumulative_fraction"] = (
    positive_damage["cumulative_damage"] / total_damage
)

# Find how many heads account for 80% of total damage
heads_to_80 = (
    positive_damage["cumulative_fraction"] < 0.80
).sum() + 1

print("=== CONCENTRATION METRIC ===")
print(f"Total heads tested: {len(results_df)}")
print(f"Heads causing positive damage: {len(positive_damage)}")
print(f"Heads needed for 80% of total damage: {heads_to_80}")

print("\nTop heads contributing to damage:")
print(
    positive_damage.head(10)[
        ["layer", "head", "margin_drop", "cumulative_fraction"]
    ]
)

In [ ]:
# Sort heads by damage
positive_damage = results_df[results_df["margin_drop"] > 0] \
    .sort_values("margin_drop", ascending=False) \
    .copy()

total_damage = positive_damage["margin_drop"].sum()

# Find cumulative contribution
cumulative = positive_damage["margin_drop"].cumsum() / total_damage

# Number of heads needed to reach 80%
heads_to_80 = (cumulative < 0.80).sum() + 1

print("Heads causing positive damage:", len(positive_damage))
print("Heads needed for 80% of total damage:", heads_to_80)

print("\nTop 10 heads:")
print(positive_damage[["layer", "head", "margin_drop"]].head(10))

In [ ]:
print("ablation_results exists:", "ablation_results" in globals())
print("results_df exists:", "results_df" in globals())

In [ ]:
print("final_model:", "final_model" in globals())
print("FIXED_IOI_SET:", "FIXED_IOI_SET" in globals())
print("baseline_acc:", "baseline_acc" in globals())
print("baseline_margin:", "baseline_margin" in globals())

In [ ]:
import pandas as pd

# Save the key findings we already know from the completed audit
key_results = pd.DataFrame([
    {
        "layer": 6,
        "head": 8,
        "accuracy_after_ablation": 0.76,
        "accuracy_drop_pp": 23.0,
        "margin_drop": 1.908334
    },
    {
        "layer": 7,
        "head": 5,
        "accuracy_after_ablation": 0.91,
        "accuracy_drop_pp": 8.0,
        "margin_drop": 1.071358
    }
])

key_results.to_csv(
    "/kaggle/working/key_ablation_results.csv",
    index=False
)

print("Results saved successfully!")
print(key_results)

In [ ]:
# Targeted verification of the most important head: Layer 6, Head 8

with ablate_heads(final_model, {6: [8]}):
    ablated_acc, ablated_margin, _ = get_ioi_metrics(
        final_model,
        FIXED_IOI_SET
    )

print("=== TARGETED ABLATION VERIFICATION ===")
print(f"Baseline accuracy: {baseline_acc:.1%}")
print(f"L6H8 ablated accuracy: {ablated_acc:.1%}")
print(f"Accuracy drop: {(baseline_acc - ablated_acc)*100:.1f} percentage points")

print(f"\nBaseline mean margin: {baseline_margin:.4f}")
print(f"L6H8 ablated margin: {ablated_margin:.4f}")
print(f"Margin drop: {baseline_margin - ablated_margin:.4f}")

In [ ]:
import pandas as pd

verification_result = pd.DataFrame([{
    "head": "L6H8",
    "baseline_accuracy": baseline_acc,
    "ablated_accuracy": ablated_acc,
    "accuracy_drop_pp": (baseline_acc - ablated_acc) * 100,
    "baseline_margin": baseline_margin,
    "ablated_margin": ablated_margin,
    "margin_drop": baseline_margin - ablated_margin
}])

verification_result.to_csv(
    "/kaggle/working/l6h8_verification.csv",
    index=False
)

print("Verification result saved!")
print(verification_result)

In [ ]:
# Targeted verification of Layer 7, Head 5

with ablate_heads(final_model, {7: [5]}):
    l7h5_acc, l7h5_margin, _ = get_ioi_metrics(
        final_model,
        FIXED_IOI_SET
    )

print("=== L7H5 VERIFICATION ===")
print(f"Baseline accuracy: {baseline_acc:.1%}")
print(f"L7H5 ablated accuracy: {l7h5_acc:.1%}")
print(f"Accuracy drop: {(baseline_acc - l7h5_acc)*100:.1f} percentage points")

print(f"\nBaseline mean margin: {baseline_margin:.4f}")
print(f"L7H5 ablated margin: {l7h5_margin:.4f}")
print(f"Margin drop: {baseline_margin - l7h5_margin:.4f}")

In [ ]:
import pandas as pd

l7h5_result = pd.DataFrame([{
    "head": "L7H5",
    "baseline_accuracy": baseline_acc,
    "ablated_accuracy": l7h5_acc,
    "accuracy_drop_pp": (baseline_acc - l7h5_acc) * 100,
    "baseline_margin": baseline_margin,
    "ablated_margin": l7h5_margin,
    "margin_drop": baseline_margin - l7h5_margin
}])

l7h5_result.to_csv(
    "/kaggle/working/l7h5_verification.csv",
    index=False
)

print("L7H5 verification saved!")
print(l7h5_result)

In [ ]:
# Ablate both important heads together

with ablate_heads(final_model, {
    6: [8],
    7: [5]
}):
    combined_acc, combined_margin, _ = get_ioi_metrics(
        final_model,
        FIXED_IOI_SET
    )

print("=== COMBINED ABLATION: L6H8 + L7H5 ===")
print(f"Baseline accuracy: {baseline_acc:.1%}")
print(f"Combined ablated accuracy: {combined_acc:.1%}")
print(f"Accuracy drop: {(baseline_acc - combined_acc)*100:.1f} percentage points")

print(f"\nBaseline mean margin: {baseline_margin:.4f}")
print(f"Combined ablated margin: {combined_margin:.4f}")
print(f"Margin drop: {baseline_margin - combined_margin:.4f}")

In [ ]:
combined_result = pd.DataFrame([{
    "heads": "L6H8 + L7H5",
    "baseline_accuracy": baseline_acc,
    "ablated_accuracy": combined_acc,
    "accuracy_drop_pp": (baseline_acc - combined_acc) * 100,
    "baseline_margin": baseline_margin,
    "ablated_margin": combined_margin,
    "margin_drop": baseline_margin - combined_margin
}])

combined_result.to_csv(
    "/kaggle/working/combined_ablation.csv",
    index=False
)

print("Combined ablation result saved!")
print(combined_result)

In [ ]:
import os

print("=== GQA CHECKPOINT FILES ===")

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".pt") and "gqa" in dirname.lower():
            print(os.path.join(dirname, filename))

In [ ]:
import torch
import os

GQA_CKPT = "/kaggle/input/datasets/nainikabatchu/gqa-s0-ckpts/ckpts/gqa_s0/final.pt"

gqa_checkpoint = torch.load(
    GQA_CKPT,
    map_location=device,
    weights_only=False
)

print("Checkpoint keys:", gqa_checkpoint.keys())

# Show configuration if present
if "config" in gqa_checkpoint:
    print("\nGQA configuration:")
    print(gqa_checkpoint["config"])
elif "model_config" in gqa_checkpoint:
    print("\nGQA configuration:")
    print(gqa_checkpoint["model_config"])

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from model import Transformer

# Create the GQA model using the checkpoint configuration
gqa_model = Transformer(gqa_checkpoint["config"]).to(device)

# Load the trained weights
gqa_model.load_state_dict(gqa_checkpoint["model"])
gqa_model.eval()

print("GQA model loaded successfully!")
print("Checkpoint step:", gqa_checkpoint["step"])
print("Tokens seen:", gqa_checkpoint["tokens_seen"])

# Confirm the model architecture
print("\nGQA model:")
print(gqa_model)

In [ ]:
import sys
import os

CODE_DIR = "/kaggle/input/datasets/nainikabatchu/archive"

print("Folder exists:", os.path.exists(CODE_DIR))
print("model.py exists:", os.path.exists(os.path.join(CODE_DIR, "model.py")))

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print("Code path added!")

In [ ]:
from model import Transformer

print("Transformer imported successfully!")

In [ ]:
import os
import sys

print("=== INPUT CONTENTS ===")
for root, dirs, files in os.walk("/kaggle/input"):
    if "model.py" in files:
        print("FOUND model.py at:", os.path.join(root, "model.py"))

print("\n=== PYTHON PATH ===")
for p in sys.path[:5]:
    print(p)

In [ ]:
import sys
import importlib.util
import traceback

MODEL_PATH = "/kaggle/input/datasets/nainikabatchu/archive/model.py"

try:
    spec = importlib.util.spec_from_file_location("model", MODEL_PATH)
    model_module = importlib.util.module_from_spec(spec)
    sys.modules["model"] = model_module
    spec.loader.exec_module(model_module)

    print("model.py loaded successfully!")
    print("\nAvailable relevant names:")
    print([name for name in dir(model_module)
           if "Transformer" in name or "Config" in name or "ablate" in name])

except Exception:
    print("ERROR WHILE LOADING model.py:")
    traceback.print_exc()

In [ ]:
# Get the classes from the successfully loaded model module
Transformer = model_module.Transformer
ModelConfig = model_module.ModelConfig

# Convert checkpoint configuration into the format expected by the model
gqa_config = ModelConfig(**gqa_checkpoint["config"])

# Create and load the GQA model
gqa_model = Transformer(gqa_config).to(device)
gqa_model.load_state_dict(gqa_checkpoint["model"])
gqa_model.eval()

print("GQA model loaded successfully!")
print("Checkpoint step:", gqa_checkpoint["step"])
print("Tokens seen:", gqa_checkpoint["tokens_seen"])
print("\nConfiguration:")
print(gqa_checkpoint["config"])

In [ ]:
print("FIXED_IOI_SET:", "FIXED_IOI_SET" in globals())
print("get_ioi_metrics:", "get_ioi_metrics" in globals())
print("Number of test examples:", len(FIXED_IOI_SET) if "FIXED_IOI_SET" in globals() else "Not found")

In [ ]:
# Evaluate GQA on the same 100-example IOI test set

gqa_baseline_acc, gqa_baseline_margin, gqa_results = get_ioi_metrics(
    gqa_model,
    FIXED_IOI_SET
)

print("=== GQA S0 FINAL MODEL BASELINE ===")
print(f"Accuracy: {gqa_baseline_acc:.1%}")
print(f"Mean IOI logit margin: {gqa_baseline_margin:.4f}")

In [ ]:
print("ablate_heads available:", "ablate_heads" in globals())

In [ ]:
ablate_heads = model_module.ablate_heads
print("ablate_heads restored!")

In [ ]:
# Quick GQA ablation test: Layer 0, Head 0

with ablate_heads(gqa_model, {0: [0]}):
    gqa_test_acc, gqa_test_margin, _ = get_ioi_metrics(
        gqa_model,
        FIXED_IOI_SET
    )

print("=== GQA SINGLE HEAD TEST ===")
print(f"Baseline accuracy: {gqa_baseline_acc:.1%}")
print(f"Ablated accuracy:  {gqa_test_acc:.1%}")

print(f"\nBaseline margin: {gqa_baseline_margin:.4f}")
print(f"Ablated margin:  {gqa_test_margin:.4f}")
print(f"Margin drop:     {gqa_baseline_margin - gqa_test_margin:.4f}")

In [ ]:
import pandas as pd
import time

results = []
start_time = time.time()

print("Testing 12 layers × 12 heads = 144 ablations")

for layer in range(12):
    for head in range(12):

        with ablate_heads(gqa_model, {layer: [head]}):
            acc, margin, _ = get_ioi_metrics(
                gqa_model,
                FIXED_IOI_SET
            )

        results.append({
            "layer": layer,
            "head": head,
            "accuracy": acc,
            "mean_margin": margin,
            "accuracy_drop_pp": (gqa_baseline_acc - acc) * 100,
            "margin_drop": gqa_baseline_margin - margin
        })

    print(f"Completed layer {layer}")

gqa_ablation_results = pd.DataFrame(results)

elapsed = (time.time() - start_time) / 60

print(f"\nCompleted in {elapsed:.1f} minutes")

print("\nTop 10 heads by margin drop:")
print(
    gqa_ablation_results
    .sort_values("margin_drop", ascending=False)
    .head(10)
)

In [ ]:
gqa_ablation_results.to_csv(
    "/kaggle/working/gqa_full_ablation_results.csv",
    index=False
)

print("GQA full ablation results saved!")
print("Number of heads tested:", len(gqa_ablation_results))

In [ ]:
with ablate_heads(gqa_model, {10: [4]}):
    gqa_l10h4_acc, gqa_l10h4_margin, _ = get_ioi_metrics(
        gqa_model,
        FIXED_IOI_SET
    )

print("=== GQA L10H4 VERIFICATION ===")
print(f"Baseline accuracy: {gqa_baseline_acc:.1%}")
print(f"L10H4 ablated accuracy: {gqa_l10h4_acc:.1%}")
print(f"Accuracy drop: {(gqa_baseline_acc - gqa_l10h4_acc)*100:.1f} percentage points")

print(f"\nBaseline mean margin: {gqa_baseline_margin:.4f}")
print(f"L10H4 ablated margin: {gqa_l10h4_margin:.4f}")
print(f"Margin drop: {gqa_baseline_margin - gqa_l10h4_margin:.4f}")

In [ ]:
import pandas as pd

gqa_l10h4_result = pd.DataFrame([{
    "head": "L10H4",
    "baseline_accuracy": gqa_baseline_acc,
    "ablated_accuracy": gqa_l10h4_acc,
    "accuracy_drop_pp": (gqa_baseline_acc - gqa_l10h4_acc) * 100,
    "baseline_margin": gqa_baseline_margin,
    "ablated_margin": gqa_l10h4_margin,
    "margin_drop": gqa_baseline_margin - gqa_l10h4_margin
}])

gqa_l10h4_result.to_csv(
    "/kaggle/working/gqa_l10h4_verification.csv",
    index=False
)

print("GQA L10H4 verification saved!")
print(gqa_l10h4_result)

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "MHA S0",
        "Baseline Accuracy": 99.0,
        "Baseline Margin": 3.2689,
        "Strongest Head": "L6H8",
        "Ablated Accuracy": 76.0,
        "Accuracy Drop (pp)": 23.0,
        "Margin Drop": 1.9083
    },
    {
        "Model": "GQA S0",
        "Baseline Accuracy": 100.0,
        "Baseline Margin": 3.6448,
        "Strongest Head": "L10H4",
        "Ablated Accuracy": 94.0,
        "Accuracy Drop (pp)": 6.0,
        "Margin Drop": 1.2452
    }
])

print("=== MHA vs GQA FINAL COMPARISON ===")
display(comparison)

comparison.to_csv(
    "/kaggle/working/mha_vs_gqa_comparison.csv",
    index=False
)

print("\nComparison saved successfully!")

In [ ]:
import os

print("=== ALL CHECKPOINT FILES ===")

for root, _, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".pt"):
            print(os.path.join(root, file))

In [ ]:
import os

print("=== LOOKING FOR eval_set.jsonl ===")

for root, _, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "eval_set.jsonl":
            print("FOUND:", os.path.join(root, file))

In [ ]:
import os

print("=== ALL FINAL CHECKPOINTS FOUND ===")

for root, _, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "final.pt":
            path = os.path.join(root, file)
            print(path)

In [ ]:
import os

print("=== SEARCHING FOR GQA S2 FILES ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for name in dirs + files:
        if "gqa" in name.lower() or "s2" in name.lower():
            print(os.path.join(root, name))

In [ ]:
import os

print("=== CHECKING FOR GQA S2 ===")

for root, _, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "final.pt" and "gqa_s2" in root:
            print("FOUND:", os.path.join(root, file))
            

In [ ]:
import torch

GQA_S2_PATH = "/kaggle/input/datasets/nainikabatchu/gqa-s2/ckpts/gqa_s2/final.pt"

ckpt = torch.load(
    GQA_S2_PATH,
    map_location="cpu",
    weights_only=False
)

print("=== CHECKPOINT KEYS ===")
print(ckpt.keys())

print("\n=== CONFIGURATION ===")
print(ckpt.get("config"))

print("\n=== RUN CONFIG ===")
print(ckpt.get("run_config"))

print("\n=== TRAINING INFO ===")
print("Step:", ckpt.get("step"))
print("Tokens seen:", ckpt.get("tokens_seen"))

In [64]:
CHECKPOINTS = {
    "mha_s0": "/kaggle/input/datasets/nainikabatchu/mha-s0-ckpts2/ckpts/mha_s0/final.pt",
    "mha_s1": "/kaggle/input/datasets/nainikabatchu/mha-s1-ckpts/ckpts/mha_s1/final.pt",
    "mha_s2": "/kaggle/input/datasets/nainikabatchu/mha-s2-ckpts/ckpts/mha_s2/final.pt",
    "gqa_s0": "/kaggle/input/datasets/nainikabatchu/gqa-s0-ckpts/ckpts/gqa_s0/final.pt",
    "gqa_s1": "/kaggle/input/datasets/nainikabatchu/gqa-s1-ckpts/ckpts/gqa_s1/final.pt",
    "gqa_s2": "/kaggle/input/datasets/nainikabatchu/gqa-s2/ckpts/gqa_s2/final.pt",
}

import os
import torch

print("=== SIX-MODEL CHECKPOINT INVENTORY ===")

for name, path in CHECKPOINTS.items():
    exists = os.path.exists(path)
    print(f"\n{name}: {'FOUND' if exists else 'MISSING'}")

    if name == "mha_s1" and not exists:
        raise FileNotFoundError("Attach version 1 of mha-s1-ckpts. Expected: " + path)

    if exists:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        config = ckpt.get("config", {})
        if name == "mha_s1" and (config.get("n_heads"), config.get("n_kv_heads")) != (12, 12):
            raise ValueError("mha_s1 requires 12 query heads and 12 KV heads. Check the attached dataset version.")
        print("  Step:", ckpt.get("step"))
        print("  d_model:", config.get("d_model"))
        print("  n_layers:", config.get("n_layers"))
        print("  n_heads:", config.get("n_heads"))
        print("  n_kv_heads:", config.get("n_kv_heads"))

=== SIX-MODEL CHECKPOINT INVENTORY ===

mha_s0: FOUND
  Step: 1907
  d_model: 768
  n_layers: 12
  n_heads: 12
  n_kv_heads: 12

mha_s1: FOUND
  Step: 1907
  d_model: 768
  n_layers: 12
  n_heads: 12
  n_kv_heads: 12

mha_s2: FOUND
  Step: 1907
  d_model: 768
  n_layers: 12
  n_heads: 12
  n_kv_heads: 12

gqa_s0: FOUND
  Step: 1907
  d_model: 768
  n_layers: 12
  n_heads: 12
  n_kv_heads: 3

gqa_s1: FOUND
  Step: 1907
  d_model: 768
  n_layers: 12
  n_heads: 12
  n_kv_heads: 3

gqa_s2: FOUND
  Step: 1907
  d_model: 768
  n_layers: 12
  n_heads: 12
  n_kv_heads: 3


In [ ]:
print("tokenizer:", "tokenizer" in globals())
print("Existing FIXED_IOI_SET size:",
      len(FIXED_IOI_SET) if "FIXED_IOI_SET" in globals() else "Not available")

In [ ]:
import json
import os

TOKENIZER_PATH = "/kaggle/input/datasets/nainikabatchu/tokenizer-32000-json/tokenizer_32000.json"

print("Tokenizer file exists:", os.path.exists(TOKENIZER_PATH))

In [ ]:
print("Tokenizer available:", "tokenizer" in globals())

if "tokenizer" in globals():
    print("Vocabulary size:", tokenizer.get_vocab_size() if hasattr(tokenizer, "get_vocab_size") else "Loaded")

In [ ]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/nainikabatchu/tokenizer-32000-json/tokenizer_32000.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.get_vocab_size())

In [ ]:
print("FIXED_IOI_SET:", "FIXED_IOI_SET" in globals())
print("get_ioi_metrics:", "get_ioi_metrics" in globals())
print("make IOI function candidates:")

for name in globals():
    if "ioi" in name.lower():
        print(" -", name)

In [ ]:
import inspect

print("=== generate_ioi_prompt signature ===")
print(inspect.signature(generate_ioi_prompt))

print("\n=== Sample generated prompt ===")
sample = generate_ioi_prompt()
print(sample)

In [ ]:
import random
import json

# Make the larger evaluation set reproducible
random.seed(42)

# Generate exactly 500 fixed IOI examples
FIXED_IOI_500 = [generate_ioi_prompt() for _ in range(500)]

print("=== FIXED 500-PROMPT IOI TEST SET CREATED ===")
print("Number of examples:", len(FIXED_IOI_500))

print("\nFirst example:")
print(FIXED_IOI_500[0])

# Save the exact evaluation set so it can be reused
with open("/kaggle/working/fixed_ioi_500.json", "w") as f:
    json.dump(FIXED_IOI_500, f, indent=2)

print("\nSaved to: /kaggle/working/fixed_ioi_500.json")

In [ ]:
print("model_module available:", "model_module" in globals())
print("device available:", "device" in globals())

In [ ]:
import sys
import importlib.util
import traceback

MODEL_PATH = "/kaggle/input/datasets/nainikabatchu/archive/model.py"

try:
    spec = importlib.util.spec_from_file_location("model", MODEL_PATH)
    model_module = importlib.util.module_from_spec(spec)
    sys.modules["model"] = model_module
    spec.loader.exec_module(model_module)

    print("model.py loaded successfully!")
    print("Relevant names:",
          [name for name in dir(model_module)
           if name in ["ModelConfig", "Transformer", "ablate_heads"]])

except Exception:
    print("ERROR WHILE LOADING model.py:")
    traceback.print_exc()

In [ ]:
import inspect

print(inspect.getsource(model_module.ablate_heads))

In [ ]:
import torch

Transformer = model_module.Transformer
ModelConfig = model_module.ModelConfig

MHA_S0_PATH = CHECKPOINTS["mha_s0"]

mha_s0_ckpt = torch.load(
    MHA_S0_PATH,
    map_location=device,
    weights_only=False
)

mha_s0_config = ModelConfig(**mha_s0_ckpt["config"])

mha_s0_model = Transformer(mha_s0_config).to(device)
mha_s0_model.load_state_dict(mha_s0_ckpt["model"])
mha_s0_model.eval()

print("MHA S0 loaded successfully!")
print("Step:", mha_s0_ckpt["step"])
print("Tokens seen:", mha_s0_ckpt["tokens_seen"])

In [ ]:
# Evaluate MHA S0 on the fixed 500-prompt IOI test set

mha_s0_baseline_acc, mha_s0_baseline_margin, mha_s0_results = get_ioi_metrics(
    mha_s0_model,
    FIXED_IOI_500
)

print("=== MHA S0: 500-PROMPT BASELINE ===")
print(f"Accuracy: {mha_s0_baseline_acc:.2%}")
print(f"Mean IOI logit margin: {mha_s0_baseline_margin:.4f}")
print(f"Number of prompts: {len(FIXED_IOI_500)}")

In [ ]:
import torch

# Store one head_out activation to inspect its shape
captured_output = {}

def inspect_hook(module, inputs, output):
    captured_output["shape"] = output.shape
    captured_output["dtype"] = output.dtype
    captured_output["device"] = output.device

# Attach hook to layer 0
handle = mha_s0_model.layers[0].attn.head_out.register_forward_hook(inspect_hook)

# Run one IOI prompt through the model
example = FIXED_IOI_500[0]
encoded = tokenizer.encode(example["prompt"])
input_ids = torch.tensor(
    [encoded.ids],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    mha_s0_model(input_ids)

handle.remove()

print("=== HEAD OUTPUT INSPECTION ===")
print("Shape:", captured_output["shape"])
print("Dtype:", captured_output["dtype"])
print("Device:", captured_output["device"])

In [ ]:
import torch

# Accumulate sums and counts for each layer and head
# MHA S0: 12 layers, 12 heads, 64 dimensions per head

head_sums = {}
head_counts = {}

for layer_idx in range(12):
    head_sums[layer_idx] = torch.zeros(
        12, 64, device=device
    )
    head_counts[layer_idx] = 0


# Create hooks to collect head outputs
handles = []

def make_mean_collection_hook(layer_idx):
    def hook(module, inputs, output):
        # output shape: [batch, heads, seq_len, head_dim]
        # Sum across batch and sequence positions
        head_sums[layer_idx] += output.detach().sum(dim=(0, 2))
        head_counts[layer_idx] += (
            output.shape[0] * output.shape[2]
        )
    return hook


for layer_idx in range(12):
    handle = mha_s0_model.layers[
        layer_idx
    ].attn.head_out.register_forward_hook(
        make_mean_collection_hook(layer_idx)
    )
    handles.append(handle)


print("Collecting reference activations from 500 prompts...")

with torch.no_grad():
    for i, example in enumerate(FIXED_IOI_500):
        encoded = tokenizer.encode(example["prompt"])
        input_ids = torch.tensor(
            [encoded.ids],
            dtype=torch.long,
            device=device
        )
        mha_s0_model(input_ids)

        if (i + 1) % 100 == 0:
            print(f"Completed {i + 1}/500 prompts")


# Always remove hooks after collection
for handle in handles:
    handle.remove()


# Compute mean vector for each head
MHA_S0_HEAD_MEANS = {}

for layer_idx in range(12):
    MHA_S0_HEAD_MEANS[layer_idx] = (
        head_sums[layer_idx] / head_counts[layer_idx]
    )

print("\n=== MEAN ACTIVATIONS COMPUTED ===")
print("Layer 0 mean shape:", MHA_S0_HEAD_MEANS[0].shape)
print("Expected: [12, 64]")

In [ ]:
from contextlib import contextmanager
import torch

@contextmanager
def mean_ablate_heads(model, heads_by_layer, head_means):
    """
    Temporarily replace selected query-head outputs with their
    reference mean activation vectors.

    Parameters
    ----------
    model : Transformer
        Model being evaluated.
    heads_by_layer : dict
        Example: {6: [8], 7: [5]}
    head_means : dict
        Precomputed mean activations for each layer.
        Each tensor has shape [n_heads, head_dim].
    """
    handles = []

    for layer_idx, heads in heads_by_layer.items():
        heads = list(heads)
        layer_means = head_means[layer_idx]

        def hook(module, inputs, output,
                 heads=heads, layer_means=layer_means):

            output = output.clone()

            # output: [batch, heads, seq_len, head_dim]
            # Replace each selected head with its mean vector
            for head_idx in heads:
                mean_vector = layer_means[head_idx].to(
                    device=output.device,
                    dtype=output.dtype
                )

                output[:, head_idx, :, :] = mean_vector.view(
                    1, 1, -1
                )

            return output

        handle = model.layers[
            layer_idx
        ].attn.head_out.register_forward_hook(hook)

        handles.append(handle)

    try:
        yield
    finally:
        for handle in handles:
            handle.remove()


print("mean_ablate_heads created successfully!")

In [ ]:
with mean_ablate_heads(
    mha_s0_model,
    {6: [8]},
    MHA_S0_HEAD_MEANS
):
    mean_test_acc, mean_test_margin, _ = get_ioi_metrics(
        mha_s0_model,
        FIXED_IOI_500
    )

print("=== MHA S0 MEAN ABLATION TEST: L6H8 ===")
print(f"Baseline accuracy: {mha_s0_baseline_acc:.2%}")
print(f"Ablated accuracy:  {mean_test_acc:.2%}")
print(f"Accuracy drop:     {(mha_s0_baseline_acc - mean_test_acc) * 100:.2f} pp")

print()
print(f"Baseline margin: {mha_s0_baseline_margin:.4f}")
print(f"Ablated margin:  {mean_test_margin:.4f}")
print(f"Margin drop:     {mha_s0_baseline_margin - mean_test_margin:.4f}")

In [ ]:
import torch

def compute_head_means(model, eval_set, n_layers=12, n_heads=12, head_dim=64):
    """
    Compute the reference mean activation vector for every
    attention head in every layer.

    Returns:
        dict[layer_idx] -> tensor of shape [n_heads, head_dim]
    """

    head_sums = {
        layer_idx: torch.zeros(
            n_heads, head_dim, device=device
        )
        for layer_idx in range(n_layers)
    }

    head_counts = {
        layer_idx: 0
        for layer_idx in range(n_layers)
    }

    handles = []

    def make_hook(layer_idx):
        def hook(module, inputs, output):
            # output shape: [batch, heads, seq_len, head_dim]
            head_sums[layer_idx] += output.detach().sum(dim=(0, 2))
            head_counts[layer_idx] += (
                output.shape[0] * output.shape[2]
            )
        return hook

    # Attach collection hooks to every layer
    for layer_idx in range(n_layers):
        handle = model.layers[
            layer_idx
        ].attn.head_out.register_forward_hook(
            make_hook(layer_idx)
        )
        handles.append(handle)

    # Run all reference prompts
    with torch.no_grad():
        for i, example in enumerate(eval_set):
            encoded = tokenizer.encode(example["prompt"])

            input_ids = torch.tensor(
                [encoded.ids],
                dtype=torch.long,
                device=device
            )

            model(input_ids)

            if (i + 1) % 100 == 0:
                print(
                    f"  Reference pass: {i + 1}/{len(eval_set)}"
                )

    # Remove hooks
    for handle in handles:
        handle.remove()

    # Calculate means
    head_means = {
        layer_idx: (
            head_sums[layer_idx] / head_counts[layer_idx]
        )
        for layer_idx in range(n_layers)
    }

    return head_means


print("Reusable compute_head_means() function created.")

In [ ]:
def load_model_from_checkpoint(checkpoint_path):
    """
    Load a Transformer model from one of the six checkpoints.
    Returns the model and checkpoint metadata.
    """
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    config = ModelConfig(**checkpoint["config"])

    model = Transformer(config).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()

    print("Model loaded successfully!")
    print("Step:", checkpoint.get("step"))
    print("Tokens seen:", checkpoint.get("tokens_seen"))
    print("n_heads:", checkpoint["config"].get("n_heads"))
    print("n_kv_heads:", checkpoint["config"].get("n_kv_heads"))

    return model, checkpoint


print("Reusable load_model_from_checkpoint() function created.")

In [ ]:
def evaluate_mean_ablation(
    model,
    eval_set,
    baseline_acc,
    baseline_margin,
    heads_by_layer,
    head_means
):
    """
    Evaluate a mean ablation and return accuracy and margin changes.
    """

    with mean_ablate_heads(model, heads_by_layer, head_means):
        ablated_acc, ablated_margin, _ = get_ioi_metrics(
            model,
            eval_set
        )

    return {
        "accuracy": ablated_acc,
        "mean_margin": ablated_margin,
        "accuracy_drop_pp": (baseline_acc - ablated_acc) * 100,
        "margin_drop": baseline_margin - ablated_margin,
    }


print("Reusable mean-ablation evaluation function created.")

In [ ]:
test_result = evaluate_mean_ablation(
    model=mha_s0_model,
    eval_set=FIXED_IOI_500,
    baseline_acc=mha_s0_baseline_acc,
    baseline_margin=mha_s0_baseline_margin,
    heads_by_layer={6: [8]},
    head_means=MHA_S0_HEAD_MEANS
)

print("=== REUSABLE ABLATION FUNCTION TEST: MHA S0 L6H8 ===")

for key, value in test_result.items():
    if key == "accuracy":
        print(f"{key}: {value:.2%}")
    else:
        print(f"{key}: {value:.4f}")

In [ ]:
import pandas as pd
import time

def run_single_head_audit(
    model,
    model_name,
    eval_set,
    baseline_acc,
    baseline_margin,
    head_means,
    n_layers=12,
    n_heads=12
):
    """
    Run mean ablation on every individual query head.
    Returns a DataFrame ranked by margin drop.
    """

    results = []
    start_time = time.time()

    print(f"=== STARTING SINGLE-HEAD AUDIT: {model_name} ===")
    print(f"Testing {n_layers} layers × {n_heads} heads = {n_layers * n_heads} ablations")

    for layer_idx in range(n_layers):
        for head_idx in range(n_heads):

            result = evaluate_mean_ablation(
                model=model,
                eval_set=eval_set,
                baseline_acc=baseline_acc,
                baseline_margin=baseline_margin,
                heads_by_layer={layer_idx: [head_idx]},
                head_means=head_means
            )

            results.append({
                "model": model_name,
                "layer": layer_idx,
                "head": head_idx,
                **result
            })

        print(f"Completed layer {layer_idx}")

    elapsed_minutes = (time.time() - start_time) / 60

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(
        "margin_drop",
        ascending=False
    ).reset_index(drop=True)

    print(f"\nCompleted in {elapsed_minutes:.1f} minutes")

    return results_df


print("Single-head audit function created.")

In [ ]:
mha_s0_single_results = run_single_head_audit(
    model=mha_s0_model,
    model_name="mha_s0",
    eval_set=FIXED_IOI_500,
    baseline_acc=mha_s0_baseline_acc,
    baseline_margin=mha_s0_baseline_margin,
    head_means=MHA_S0_HEAD_MEANS
)

# Save immediately
OUTPUT_PATH = "/kaggle/working/mha_s0_single_head_results.csv"

mha_s0_single_results.to_csv(OUTPUT_PATH, index=False)

print("\nResults saved successfully!")
print("Saved to:", OUTPUT_PATH)

print("\n=== TOP 10 HEADS BY MARGIN DROP ===")
print(mha_s0_single_results.head(10))

In [ ]:
print("Session is active")

In [ ]:
print("mha_s0_single_results" in globals())


In [ ]:
import os

print(os.listdir("/kaggle/working"))

In [ ]:
print("tokenizer:", "tokenizer" in globals())
print("FIXED_IOI_500:", "FIXED_IOI_500" in globals())
print("get_ioi_metrics:", "get_ioi_metrics" in globals())
print("mean_ablate_heads:", "mean_ablate_heads" in globals())

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
print("tokenizer:", "tokenizer" in globals())
print("FIXED_IOI_500:", "FIXED_IOI_500" in globals())
print("get_ioi_metrics:", "get_ioi_metrics" in globals())
print("ModelConfig:", "ModelConfig" in globals())
print("Transformer:", "Transformer" in globals())

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")

In [ ]:
import os
import pandas as pd
import time

def run_grouped_audit(
    model,
    model_name,
    eval_set,
    baseline_acc,
    baseline_margin,
    head_means,
    output_dir="/kaggle/working"
):
    """
    Tests 3 groups per layer:
      Group 0: heads 0-3
      Group 1: heads 4-7
      Group 2: heads 8-11

    Saves progress after every layer and resumes automatically.
    """

    output_path = os.path.join(
        output_dir,
        f"{model_name}_grouped_results.csv"
    )

    groups = {
        0: [0, 1, 2, 3],
        1: [4, 5, 6, 7],
        2: [8, 9, 10, 11]
    }

    if os.path.exists(output_path):
        results_df = pd.read_csv(output_path)
        results = results_df.to_dict("records")
        completed_layers = set(results_df["layer"].unique())
        print(f"Resuming {model_name}. Completed layers: {sorted(completed_layers)}")
    else:
        results = []
        completed_layers = set()
        print(f"Starting new audit: {model_name}")

    start_time = time.time()

    for layer_idx in range(12):

        if layer_idx in completed_layers:
            continue

        for group_idx, heads in groups.items():

            result = evaluate_mean_ablation(
                model=model,
                eval_set=eval_set,
                baseline_acc=baseline_acc,
                baseline_margin=baseline_margin,
                heads_by_layer={layer_idx: heads},
                head_means=head_means
            )

            results.append({
                "model": model_name,
                "layer": layer_idx,
                "group": group_idx,
                "heads": str(heads),
                **result
            })

        # Save after every layer
        pd.DataFrame(results).to_csv(output_path, index=False)
        print(f"Completed and saved layer {layer_idx}")

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(
        "margin_drop",
        ascending=False
    ).reset_index(drop=True)

    results_df.to_csv(output_path, index=False)

    elapsed = (time.time() - start_time) / 60

    print(f"\n=== {model_name} COMPLETE ===")
    print(f"Time: {elapsed:.1f} minutes")
    print("\nTop groups by margin drop:")
    print(results_df.head(10))

    return results_df


print("Resumable grouped audit function ready.")

In [ ]:
print("evaluate_mean_ablation:", "evaluate_mean_ablation" in globals())
print("compute_head_means:", "compute_head_means" in globals())

In [ ]:
MHA_S0_CKPT = "/kaggle/input/datasets/nainikabatchu/mha-s0-ckpts2/ckpts/mha_s0/final.pt"

In [ ]:
# MHA S0 checkpoint
MHA_S0_CKPT = "/kaggle/input/datasets/nainikabatchu/mha-s0-ckpts2/ckpts/mha_s0/final.pt"

# 1. Load model
mha_s0_model, mha_s0_checkpoint = load_model_from_checkpoint(
    MHA_S0_CKPT
)

# 2. Calculate baseline
mha_s0_baseline_acc, mha_s0_baseline_margin, _ = get_ioi_metrics(
    mha_s0_model,
    FIXED_IOI_500
)

print("\n=== MHA S0 BASELINE ===")
print(f"Accuracy: {mha_s0_baseline_acc:.2%}")
print(f"Mean margin: {mha_s0_baseline_margin:.4f}")

# 3. Compute reference mean activations
print("\nComputing mean head activations...")
MHA_S0_HEAD_MEANS = compute_head_means(
    mha_s0_model,
    FIXED_IOI_500
)

print("Mean activations ready.")

# 4. Run grouped audit
mha_s0_grouped_results = run_grouped_audit(
    model=mha_s0_model,
    model_name="mha_s0",
    eval_set=FIXED_IOI_500,
    baseline_acc=mha_s0_baseline_acc,
    baseline_margin=mha_s0_baseline_margin,
    head_means=MHA_S0_HEAD_MEANS
)

In [ ]:
CHECKPOINTS = {
    "mha_s1": "/kaggle/input/datasets/nainikabatchu/mha-s1-ckpts/ckpts/mha_s1/final.pt",
    "mha_s2": "/kaggle/input/datasets/nainikabatchu/mha-s2-ckpts/ckpts/mha_s2/final.pt",
    "gqa_s0": "/kaggle/input/datasets/nainikabatchu/gqa-s0-ckpts/ckpts/gqa_s0/final.pt",
    "gqa_s1": "/kaggle/input/datasets/nainikabatchu/gqa-s1-ckpts/ckpts/gqa_s1/final.pt",
    "gqa_s2": "/kaggle/input/datasets/nainikabatchu/gqa-s2/ckpts/gqa_s2/final.pt",
}

for name, path in CHECKPOINTS.items():
    print(name, "→", os.path.exists(path))

In [ ]:
all_grouped_results = {
    "mha_s0": mha_s0_grouped_results
}

all_baselines = {
    "mha_s0": {
        "accuracy": mha_s0_baseline_acc,
        "margin": mha_s0_baseline_margin
    }
}

for model_name, checkpoint_path in CHECKPOINTS.items():

    print("\n" + "=" * 60)
    print(f"PROCESSING {model_name.upper()}")
    print("=" * 60)

    # Load model
    current_model, current_checkpoint = load_model_from_checkpoint(
        checkpoint_path
    )

    # Baseline
    baseline_acc, baseline_margin, _ = get_ioi_metrics(
        current_model,
        FIXED_IOI_500
    )

    print(f"\nBaseline accuracy: {baseline_acc:.2%}")
    print(f"Baseline margin: {baseline_margin:.4f}")

    # Compute model-specific mean activations
    print("\nComputing mean head activations...")
    current_head_means = compute_head_means(
        current_model,
        FIXED_IOI_500
    )

    # Run grouped audit
    grouped_results = run_grouped_audit(
        model=current_model,
        model_name=model_name,
        eval_set=FIXED_IOI_500,
        baseline_acc=baseline_acc,
        baseline_margin=baseline_margin,
        head_means=current_head_means
    )

    # Store results
    all_grouped_results[model_name] = grouped_results

    all_baselines[model_name] = {
        "accuracy": baseline_acc,
        "margin": baseline_margin
    }

    # Free GPU memory before the next model
    del current_model
    torch.cuda.empty_cache()

print("\n=== ALL SIX-MODEL GROUPED AUDITS COMPLETE ===")

In [ ]:
import os
import pandas as pd

def find_components_to_80(
    model,
    model_name,
    grouped_results,
    eval_set,
    baseline_margin,
    head_means,
    output_dir="/kaggle/working"
):
    """
    Rank groups by individual margin drop, then cumulatively ablate
    the top groups until >=80% of the baseline margin is lost.
    Saves progress after every cumulative step.
    """

    ranked = grouped_results.sort_values(
        "margin_drop",
        ascending=False
    ).reset_index(drop=True)

    output_path = os.path.join(
        output_dir,
        f"{model_name}_cumulative_ablation.csv"
    )

    # Each entry maps a layer to the heads that should be ablated
    cumulative_heads = {}
    cumulative_results = []

    target_damage = 0.80 * baseline_margin

    for k, (_, row) in enumerate(ranked.iterrows(), start=1):

        layer = int(row["layer"])

        # Convert the saved string "[0, 1, 2, 3]" back into a list
        heads = eval(row["heads"])

        if layer not in cumulative_heads:
            cumulative_heads[layer] = []

        cumulative_heads[layer].extend(heads)

        # Remove duplicates just in case
        cumulative_heads[layer] = sorted(
            set(cumulative_heads[layer])
        )

        result = evaluate_mean_ablation(
            model=model,
            eval_set=eval_set,
            baseline_acc=0.0,  # Accuracy isn't the primary measure here
            baseline_margin=baseline_margin,
            heads_by_layer=cumulative_heads,
            head_means=head_means
        )

        damage_fraction = result["margin_drop"] / baseline_margin

        cumulative_results.append({
            "model": model_name,
            "k_components": k,
            "layer_added": layer,
            "group_added": int(row["group"]),
            "individual_margin_drop": row["margin_drop"],
            "cumulative_margin": result["mean_margin"],
            "cumulative_margin_drop": result["margin_drop"],
            "damage_fraction": damage_fraction
        })

        # Save after every step
        pd.DataFrame(cumulative_results).to_csv(
            output_path,
            index=False
        )

        print(
            f"{model_name}: k={k}, "
            f"damage={damage_fraction:.1%}"
        )

        if result["margin_drop"] >= target_damage:
            print(
                f"\n80% damage reached at "
                f"k = {k} components"
            )
            break

    return pd.DataFrame(cumulative_results)

In [ ]:
print("MHA S0 model:", "mha_s0_model" in globals())
print("MHA S0 means:", "MHA_S0_HEAD_MEANS" in globals())
print("MHA S0 results:", "mha_s0_grouped_results" in globals())

In [ ]:
mha_s0_cumulative = find_components_to_80(
    model=mha_s0_model,
    model_name="mha_s0",
    grouped_results=mha_s0_grouped_results,
    eval_set=FIXED_IOI_500,
    baseline_margin=mha_s0_baseline_margin,
    head_means=MHA_S0_HEAD_MEANS
)

print("\n=== MHA S0 CUMULATIVE RESULTS ===")
print(mha_s0_cumulative)

In [ ]:
# Paths for the remaining models
REMAINING_CHECKPOINTS = {
    "mha_s1": "/kaggle/input/datasets/nainikabatchu/mha-s1-ckpts/ckpts/mha_s1/final.pt",
    "mha_s2": "/kaggle/input/datasets/nainikabatchu/mha-s2-ckpts/ckpts/mha_s2/final.pt",
    "gqa_s0": "/kaggle/input/datasets/nainikabatchu/gqa-s0-ckpts/ckpts/gqa_s0/final.pt",
    "gqa_s1": "/kaggle/input/datasets/nainikabatchu/gqa-s1-ckpts/ckpts/gqa_s1/final.pt",
    "gqa_s2": "/kaggle/input/datasets/nainikabatchu/gqa-s2/ckpts/gqa_s2/final.pt",
}

# Load grouped results and baseline information
baseline_lookup = all_baselines.copy()

print("Models ready for cumulative analysis:")
for name in REMAINING_CHECKPOINTS:
    csv_path = f"/kaggle/working/{name}_grouped_results.csv"
    print(name, "→ grouped CSV exists:", os.path.exists(csv_path))

In [ ]:
remaining_cumulative = {}

for model_name, checkpoint_path in REMAINING_CHECKPOINTS.items():

    print("\n" + "=" * 60)
    print(f"CUMULATIVE ANALYSIS: {model_name.upper()}")
    print("=" * 60)

    # Load model
    current_model, _ = load_model_from_checkpoint(checkpoint_path)

    # Get this model's baseline values
    baseline_acc = baseline_lookup[model_name]["accuracy"]
    baseline_margin = baseline_lookup[model_name]["margin"]

    # Load its grouped results
    grouped_results = pd.read_csv(
        f"/kaggle/working/{model_name}_grouped_results.csv"
    )

    # Compute model-specific mean activations
    print("Computing reference mean activations...")
    current_head_means = compute_head_means(
        current_model,
        FIXED_IOI_500
    )

    # Run cumulative ablation
    cumulative_results = find_components_to_80(
        model=current_model,
        model_name=model_name,
        grouped_results=grouped_results,
        eval_set=FIXED_IOI_500,
        baseline_margin=baseline_margin,
        head_means=current_head_means
    )

    remaining_cumulative[model_name] = cumulative_results

    print("\nFinal cumulative result:")
    print(cumulative_results.tail(1))

    # Free GPU memory
    del current_model
    del current_head_means
    torch.cuda.empty_cache()

print("\n=== ALL REMAINING CUMULATIVE ANALYSES COMPLETE ===")

In [ ]:
# Build the final concentration summary

concentration_rows = []

# MHA S0 was run separately
all_cumulative = {
    "mha_s0": mha_s0_cumulative,
    **remaining_cumulative
}

for model_name, cumulative_df in all_cumulative.items():

    # Number of components needed to reach 80% damage
    components_to_80 = int(
        cumulative_df.iloc[-1]["k_components"]
    )

    # Top component score = damage fraction from the strongest component
    top_component_score = float(
        cumulative_df.iloc[0]["damage_fraction"]
    )

    concentration_rows.append({
        "model": model_name,
        "heads_to_80": components_to_80,
        "top_head_score": top_component_score
    })

concentration_scores = pd.DataFrame(
    concentration_rows
).sort_values("model").reset_index(drop=True)

# Save the required file
output_path = "/kaggle/working/concentration_scores.csv"
concentration_scores.to_csv(output_path, index=False)

print("=== CONCENTRATION SCORES ===")
print(concentration_scores.to_string(index=False))

print(f"\nSaved to: {output_path}")

In [ ]:
import os

print("=== SEARCHING FOR SFT / SKT CHECKPOINTS AND EVALUATION FILES ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        path = os.path.join(root, file)
        name = file.lower()

        if (
            "sft" in name
            or "eval_set" in name
            or "ag_news" in name
            or "agnews" in name
        ):
            print(path)

In [ ]:
import os

print("=== ALL SFT CHECKPOINT FILES ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "final.pt" or file.endswith(".jsonl"):
            path = os.path.join(root, file)

            # Show only likely SFT-related files
            if "sft" in path.lower():
                print(path)

In [ ]:
print("\n=== SEARCHING FOR EVAL_SET.JSONL ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower() == "eval_set.jsonl":
            print("FOUND:", os.path.join(root, file))

In [ ]:
import json

sample_path = "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s0-datset/preds_mha_s0.jsonl"

print("=== FIRST 3 RECORDS ===")

with open(sample_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(json.loads(line))
        

In [ ]:
import os

print("=== ALL JSONL FILES IN KAGGLE INPUTS ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".jsonl"):
            print(os.path.join(root, file))

In [ ]:
import os

print("=== CSV FILES IN /kaggle/working ===")

for file in sorted(os.listdir("/kaggle/working")):
    if file.endswith(".csv"):
        print(file)

In [ ]:
import pandas as pd

# Load concentration scores
concentration = pd.read_csv(
    "/kaggle/working/concentration_scores.csv"
)

# Add architecture and seed information
model_info = {
    "mha_s0": {"architecture": "MHA", "seed": 0},
    "mha_s1": {"architecture": "MHA", "seed": 1},
    "mha_s2": {"architecture": "MHA", "seed": 2},
    "gqa_s0": {"architecture": "GQA", "seed": 0},
    "gqa_s1": {"architecture": "GQA", "seed": 1},
    "gqa_s2": {"architecture": "GQA", "seed": 2},
}

info_df = pd.DataFrame([
    {"model": model, **details}
    for model, details in model_info.items()
])

# Combine everything
summary_df = concentration.merge(info_df, on="model")

# Arrange columns
summary_df = summary_df[
    ["model", "architecture", "seed", "heads_to_80", "top_head_score"]
].sort_values(["architecture", "seed"])

print("=== SIX-MODEL CONCENTRATION SUMMARY ===")
display(summary_df)

# Save for later use
summary_df.to_csv(
    "/kaggle/working/six_model_concentration_summary.csv",
    index=False
)

print("\nSaved to: /kaggle/working/six_model_concentration_summary.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

models = ["mha_s0", "mha_s1", "mha_s2",
          "gqa_s0", "gqa_s1", "gqa_s2"]

for model in models:
    path = f"/kaggle/working/{model}_grouped_results.csv"
    df = pd.read_csv(path)

    # Create a layer × group matrix using margin drop
    heatmap_data = df.pivot(
        index="layer",
        columns="group",
        values="margin_drop"
    )

    plt.figure(figsize=(7, 6))
    plt.imshow(heatmap_data, aspect="auto")

    plt.colorbar(label="Mean logit margin drop")
    plt.xlabel("Component / KV group")
    plt.ylabel("Layer")
    plt.title(f"{model.upper()} — Component Ablation Impact")

    plt.xticks(
        range(len(heatmap_data.columns)),
        [f"G{g}" for g in heatmap_data.columns]
    )
    plt.yticks(range(len(heatmap_data.index)), heatmap_data.index)

    plt.tight_layout()

    output_path = f"/kaggle/working/{model}_heatmap.png"
    plt.savefig(output_path, dpi=200)
    plt.show()
    plt.close()

    print(f"Saved: {output_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

models = [
    "mha_s0", "mha_s1", "mha_s2",
    "gqa_s0", "gqa_s1", "gqa_s2"
]

plt.figure(figsize=(9, 6))

for model in models:
    path = f"/kaggle/working/{model}_cumulative_ablation.csv"
    df = pd.read_csv(path)

    plt.plot(
        df["k_components"],
        df["damage_fraction"] * 100,
        marker="o",
        label=model.upper()
    )

# Show the 80% damage threshold
plt.axhline(80, linestyle="--", label="80% damage threshold")

plt.xlabel("Number of components cumulatively ablated")
plt.ylabel("Baseline margin damage (%)")
plt.title("Cumulative Functional Damage Across Models")
plt.xticks(range(1, 4))
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "/kaggle/working/cumulative_damage_all_models.png",
    dpi=200
)
plt.show()

print("Saved: /kaggle/working/cumulative_damage_all_models.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary = pd.read_csv(
    "/kaggle/working/six_model_concentration_summary.csv"
)

# Sort for consistent display
summary = summary.sort_values(["architecture", "seed"])

plt.figure(figsize=(8, 5))

plt.bar(
    summary["model"].str.upper(),
    summary["heads_to_80"]
)

plt.xlabel("Model")
plt.ylabel("Components required for 80% damage")
plt.title("Functional Concentration Across Models")

plt.tight_layout()
plt.savefig(
    "/kaggle/working/components_to_80_comparison.png",
    dpi=200
)
plt.show()

print("Saved: /kaggle/working/components_to_80_comparison.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary = pd.read_csv(
    "/kaggle/working/six_model_concentration_summary.csv"
)

summary = summary.sort_values(["architecture", "seed"])

plt.figure(figsize=(8, 5))

plt.bar(
    summary["model"].str.upper(),
    summary["top_head_score"] * 100
)

plt.xlabel("Model")
plt.ylabel("Damage caused by strongest component (%)")
plt.title("Impact of the Strongest Functional Component")

plt.tight_layout()
plt.savefig(
    "/kaggle/working/top_component_damage_comparison.png",
    dpi=200
)
plt.show()

print("Saved: /kaggle/working/top_component_damage_comparison.png")

In [ ]:
import pandas as pd

# Load the concentration summary
summary = pd.read_csv(
    "/kaggle/working/six_model_concentration_summary.csv"
)

# Store baseline metrics from the completed evaluations
baseline_data = {
    "mha_s0": {"baseline_accuracy": 99.4, "baseline_margin": 3.5087},
    "mha_s1": {"baseline_accuracy": 98.0, "baseline_margin": 3.3409},
    "mha_s2": {"baseline_accuracy": 94.6, "baseline_margin": 2.6522},
    "gqa_s0": {"baseline_accuracy": 99.6, "baseline_margin": 3.7612},
    "gqa_s1": {"baseline_accuracy": 94.8, "baseline_margin": 3.1766},
    "gqa_s2": {"baseline_accuracy": 97.4, "baseline_margin": 3.8538},
}

baseline_df = pd.DataFrame([
    {"model": model, **values}
    for model, values in baseline_data.items()
])

# Combine baseline and concentration results
final_summary = summary.merge(baseline_df, on="model")

final_summary = final_summary[
    [
        "model",
        "architecture",
        "seed",
        "baseline_accuracy",
        "baseline_margin",
        "heads_to_80",
        "top_head_score",
    ]
].sort_values(["architecture", "seed"])

print("=== FINAL PRETRAINED MODEL RESULTS ===")
display(final_summary)

final_summary.to_csv(
    "/kaggle/working/final_pretrained_results_summary.csv",
    index=False
)

print("\nSaved: /kaggle/working/final_pretrained_results_summary.csv")

In [ ]:
import os
import shutil

output_dir = "/kaggle/working"
backup_dir = "/kaggle/working/pretrained_analysis_backup"

os.makedirs(backup_dir, exist_ok=True)

# Files and patterns to preserve
for file in os.listdir(output_dir):
    if (
        file.endswith(".csv")
        or file.endswith(".png")
    ):
        src = os.path.join(output_dir, file)
        dst = os.path.join(backup_dir, file)
        shutil.copy2(src, dst)

# Create ZIP backup
zip_path = "/kaggle/working/pretrained_analysis_backup"
shutil.make_archive(zip_path, "zip", backup_dir)

print("Backup created successfully:")
print(zip_path + ".zip")

print("\nFiles backed up:")
for file in os.listdir(backup_dir):
    print("-", file)

In [ ]:
import os

print("=== KAGGLE INPUT DATASETS ===")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level

    if level <= 3:
        print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        if file in ["eval_set.jsonl", "final.pt"]:
            print(f"{indent}  📄 {file}")

In [ ]:
with open(EVAL_PATH, "r") as f:
    total = sum(1 for _ in f)

print("Total records:", total)

In [ ]:
import os

print("=== SEARCHING FOR eval_set.jsonl ===")

found = False

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "eval_set.jsonl":
            print(os.path.join(root, file))
            found = True

if not found:
    print("NOT FOUND")

In [ ]:
import os

print("=== SEARCHING FOR eval_set.jsonl ===")

EVAL_PATH = None

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "eval_set.jsonl":
            EVAL_PATH = os.path.join(root, file)
            print("FOUND:", EVAL_PATH)

if EVAL_PATH is None:
    print("NOT FOUND")

In [ ]:
import json

with open(EVAL_PATH, "r") as f:
    records = [json.loads(line) for line in f]

print("Total records:", len(records))
print("\nFirst 3 records:")
for record in records[:3]:
    print(record)

In [ ]:
import os

print("=== SFT CHECKPOINTS ===")

sft_checkpoints = {}

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "final.pt" and "sft-runner" in root:
            path = os.path.join(root, file)
            print(path)

In [ ]:
SFT_CHECKPOINTS = {
    "mha_s0": "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s0-datset/ckpts/mha_s0_sft/final.pt",
    "mha_s1": "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s1-datset/ckpts/mha_s1_sft/final.pt",
    "mha_s2": "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s2-datset/ckpts/mha_s2_sft/final.pt",
    "gqa_s0": "/kaggle/input/datasets/nainikabatchu/sft-runner-gqa-s0-datset/ckpts/gqa_s0_sft/final.pt",
    "gqa_s1": "/kaggle/input/datasets/nainikabatchu/sft-runner-gqa-s1-datset/ckpts/gqa_s1_sft/final.pt",
    "gqa_s2": "/kaggle/input/datasets/nainikabatchu/sft-runner-gqa-s2-datset/ckpts/gqa_s2_sft/final.pt",
}

for name, path in SFT_CHECKPOINTS.items():
    print(f"{name}: {os.path.exists(path)}")

In [ ]:
print("tokenizer:", "tokenizer" in globals())
print("FIXED_IOI_500:", "FIXED_IOI_500" in globals())
print("get_ioi_metrics:", "get_ioi_metrics" in globals())
print("mean_ablate_heads:", "mean_ablate_heads" in globals())

In [ ]:
MHA_S0_SFT_PATH = SFT_CHECKPOINTS["mha_s0"]

print("Checkpoint:", MHA_S0_SFT_PATH)
print("Exists:", os.path.exists(MHA_S0_SFT_PATH))

In [ ]:
print([name for name in globals() if "load" in name.lower()])

In [ ]:
import inspect

print(inspect.signature(load_model))

In [ ]:
print(inspect.getsource(load_model))

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

mha_s0_sft_model, mha_s0_sft_info = load_model(
    SFT_CHECKPOINTS["mha_s0"],
    device=DEVICE
)

print("Model loaded successfully!")
print("Device:", DEVICE)

for key in ["step", "tokens_seen"]:
    if key in mha_s0_sft_info:
        print(f"{key}: {mha_s0_sft_info[key]}")

print("n_heads:", mha_s0_sft_model.config.n_heads)
print("n_kv_heads:", mha_s0_sft_model.config.n_kv_heads)

In [ ]:
ckpt = torch.load(
    SFT_CHECKPOINTS["mha_s0"],
    map_location="cpu",
    weights_only=False
)

print("=== MHA S0 SFT MODEL ===")
print("n_heads:", ckpt["config"]["n_heads"])
print("n_kv_heads:", ckpt["config"]["n_kv_heads"])
print("step:", ckpt.get("step"))
print("tokens_seen:", ckpt.get("tokens_seen"))

In [ ]:
print("Available evaluation-related functions:")

for name in list(globals().keys()):
    name_lower = name.lower()
    if any(word in name_lower for word in ["eval", "metric", "margin", "accuracy"]):
        if callable(globals()[name]):
            print(name)

In [ ]:
import inspect

print("evaluate_fixed_ioI:")
print(inspect.signature(evaluate_fixed_ioi))

print("\nget_ioi_metrics:")
print(inspect.signature(get_ioi_metrics))

In [ ]:
mha_s0_sft_baseline_acc, mha_s0_sft_baseline_margin = get_ioi_metrics(
    mha_s0_sft_model,
    FIXED_IOI_500
)

print("=== MHA S0 SFT BASELINE ===")
print(f"Accuracy: {mha_s0_sft_baseline_acc:.2%}")
print(f"Mean margin: {mha_s0_sft_baseline_margin:.4f}")

In [ ]:
result = get_ioi_metrics(
    mha_s0_sft_model,
    FIXED_IOI_500
)

print("Type:", type(result))
print("Number of returned values:", len(result))

for i, value in enumerate(result):
    print(f"[{i}] = {value}")

In [ ]:
mha_s0_sft_baseline_acc, mha_s0_sft_baseline_margin, mha_s0_sft_details = get_ioi_metrics(
    mha_s0_sft_model,
    FIXED_IOI_500
)

print("Baseline accuracy:", mha_s0_sft_baseline_acc)
print("Baseline margin:", mha_s0_sft_baseline_margin)

In [ ]:
# Check whether the important functions from the previous concentration audit exist
needed = [
    "compute_reference_mean_activations",
    "get_ioi_metrics",
    "FIXED_IOI_500"
]

for name in needed:
    print(name, "→", name in globals())

In [ ]:
# Find ablation/concentration-related functions
for name in list(globals()):
    lower = name.lower()
    if any(x in lower for x in ["ablat", "concentration", "reference_mean", "hook"]):
        print(name)

In [ ]:
import inspect

print(inspect.signature(mean_ablate_heads))
print("\nSOURCE:\n")
print(inspect.getsource(mean_ablate_heads))

In [ ]:
print(inspect.signature(ablate_heads))
print("\nSOURCE:\n")
print(inspect.getsource(ablate_heads))

In [ ]:
from collections import defaultdict
import torch

@torch.no_grad()
def compute_reference_mean_activations(model, test_set):
    """
    Compute mean output activation for every attention head
    using the prompts in FIXED_IOI_500.
    """
    model.eval()

    # Store activations for each layer
    collected = defaultdict(list)
    handles = []

    n_layers = len(model.layers)

    for layer_idx in range(n_layers):

        def make_hook(idx):
            def hook(module, inputs, output):
                # output shape: [batch, heads, seq_len, head_dim]
                collected[idx].append(
                    output.detach().cpu()
                )
            return hook

        handle = model.layers[
            layer_idx
        ].attn.head_out.register_forward_hook(
            make_hook(layer_idx)
        )

        handles.append(handle)

    print("Computing reference mean activations...")

    for i, ex in enumerate(test_set):
        ids = tokenizer.encode(ex["prompt"]).ids

        input_ids = torch.tensor(
            [ids],
            dtype=torch.long,
            device=device
        )

        model(input_ids)

        if (i + 1) % 100 == 0:
            print(f"  Reference pass: {i + 1}/{len(test_set)}")

    # Remove hooks
    for handle in handles:
        handle.remove()

    # Calculate mean activation per head
    head_means = {}

    for layer_idx in range(n_layers):
        # Each activation may have a different sequence length.
        # Average over batch and sequence dimensions first.
        per_example_means = []

        for activation in collected[layer_idx]:
            # [batch, heads, seq_len, head_dim]
            mean_activation = activation.mean(dim=(0, 2))
            # Result: [heads, head_dim]
            per_example_means.append(mean_activation)

        head_means[layer_idx] = torch.stack(
            per_example_means
        ).mean(dim=0)

    print("Reference means computed successfully.")
    return head_means

In [ ]:
from collections import defaultdict
import torch

@torch.no_grad()
def compute_reference_mean_activations(model, test_set):
    """
    Compute mean output activation for every attention head
    using the prompts in FIXED_IOI_500.
    """
    model.eval()

    # Store activations for each layer
    collected = defaultdict(list)
    handles = []

    n_layers = len(model.layers)

    for layer_idx in range(n_layers):

        def make_hook(idx):
            def hook(module, inputs, output):
                # output shape: [batch, heads, seq_len, head_dim]
                collected[idx].append(
                    output.detach().cpu()
                )
            return hook

        handle = model.layers[
            layer_idx
        ].attn.head_out.register_forward_hook(
            make_hook(layer_idx)
        )

        handles.append(handle)

    print("Computing reference mean activations...")

    for i, ex in enumerate(test_set):
        ids = tokenizer.encode(ex["prompt"]).ids

        input_ids = torch.tensor(
            [ids],
            dtype=torch.long,
            device=device
        )

        model(input_ids)

        if (i + 1) % 100 == 0:
            print(f"  Reference pass: {i + 1}/{len(test_set)}")

    # Remove hooks
    for handle in handles:
        handle.remove()

    # Calculate mean activation per head
    head_means = {}

    for layer_idx in range(n_layers):
        # Each activation may have a different sequence length.
        # Average over batch and sequence dimensions first.
        per_example_means = []

        for activation in collected[layer_idx]:
            # [batch, heads, seq_len, head_dim]
            mean_activation = activation.mean(dim=(0, 2))
            # Result: [heads, head_dim]
            per_example_means.append(mean_activation)

        head_means[layer_idx] = torch.stack(
            per_example_means
        ).mean(dim=0)

    print("Reference means computed successfully.")
    return head_means

In [ ]:
print("Type:", type(FIXED_IOI_500))
print("Length:", len(FIXED_IOI_500))
print("\nFirst item:")
print(FIXED_IOI_500[0])

if isinstance(FIXED_IOI_500[0], dict):
    print("\nKeys:", FIXED_IOI_500[0].keys())


In [ ]:
import inspect
print(inspect.getsource(get_ioi_metrics))

In [ ]:
mha_s0_sft_head_means = compute_reference_mean_activations(
    mha_s0_sft_model,
    FIXED_IOI_500
)

In [ ]:
# Check for variables that may contain the top components
for name in list(globals().keys()):
    if any(word in name.lower() for word in ["top", "ablation", "component", "group"]):
        print(name)

In [ ]:
print(type(ablation_results))

# If it's a DataFrame, show the MHA S0 rows
try:
    display(ablation_results[
        ablation_results["model"] == "mha_s0"
    ].head(10))
except Exception as e:
    print("Columns/structure:", e)
    print(ablation_results)

In [ ]:
import pandas as pd

# Convert the results list into a DataFrame
ablation_df = pd.DataFrame(ablation_results)

# Sort by margin damage.
# More negative margin_drop = greater damage.
top_heads = ablation_df.sort_values(
    "margin_drop",
    ascending=True
)

print("=== TOP 10 MOST IMPORTANT HEADS ===")
display(top_heads.head(10))

In [ ]:
print("Total ablation results:", len(ablation_results))

# Show the layer/head range covered
print("Layers present:", sorted(set(x["layer"] for x in ablation_results)))
print("Heads present:", sorted(set(x["head"] for x in ablation_results)))

In [ ]:
import os

print("=== LOOKING FOR PREVIOUS ABLATION CSV FILES ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
# Search all currently available variables for possible
# stored results from the previous analysis
for name in list(globals().keys()):
    if any(term in name.lower() for term in [
        "grouped", "cumulative", "key_ablation",
        "concentration", "results"
    ]):
        obj = globals()[name]
        print(f"{name}: {type(obj)}")

In [ ]:
print("=== KEY RESULTS ===")
display(key_results)

print("\n=== MHA S0 RESULTS ===")
print("Type:", type(mha_s0_results))
print("Number of entries:", len(mha_s0_results))

# Show the first few entries
for item in mha_s0_results[:10]:
    print(item)

In [ ]:
print("=== Checking other stored variables ===")

for name in ["baseline_results", "early_results", "final_results", "test_results"]:
    obj = globals().get(name)
    print(f"\n{name}:")
    print(type(obj))

    if isinstance(obj, list) and len(obj) > 0:
        print("Length:", len(obj))
        print("First item:", obj[0])

In [ ]:
import pandas as pd

print("=== SEARCHING VARIABLES FOR COMPONENT RESULTS ===")

for name, obj in list(globals().items()):
    # Check DataFrames
    if isinstance(obj, pd.DataFrame):
        cols = list(obj.columns)
        if "layer" in cols and ("head" in cols or "group" in cols):
            print(f"\n{name} — DataFrame")
            print("Columns:", cols)
            print(obj.head())

    # Check lists of dictionaries
    elif isinstance(obj, list) and len(obj) > 0 and isinstance(obj[0], dict):
        keys = obj[0].keys()
        if "layer" in keys and ("head" in keys or "group" in keys):
            print(f"\n{name} — list")
            print("Length:", len(obj))
            print("First 3:", obj[:3])

In [ ]:
import os

print("=== SEARCHING FOR PREVIOUS RESULT FILES ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv") or file.endswith(".zip"):
            if any(word in file.lower() for word in [
                "ablation", "grouped", "cumulative",
                "concentration", "result"
            ]):
                print(os.path.join(root, file))

In [ ]:
import pandas as pd
import os

RESULTS_DIR = "/kaggle/input/datasets/nainikabatchu/results"

models = [
    "mha_s0", "mha_s1", "mha_s2",
    "gqa_s0", "gqa_s1", "gqa_s2"
]

for model_name in models:
    print("\n" + "="*60)
    print(model_name.upper())
    print("="*60)

    path = os.path.join(
        RESULTS_DIR,
        f"{model_name}_cumulative_ablation.csv"
    )

    df = pd.read_csv(path)

    print("Columns:", list(df.columns))
    display(df.head(5))

In [ ]:
import pandas as pd

path = "/kaggle/input/datasets/nainikabatchu/results/mha_s0_cumulative_ablation.csv"
mha_s0_cumulative = pd.read_csv(path)

display(mha_s0_cumulative)

In [ ]:
print("Layers:", len(mha_s0_sft_model.layers))
print("Heads:", mha_s0_sft_info["config"]["n_heads"])
print("KV heads:", mha_s0_sft_info["config"]["n_kv_heads"])

In [ ]:
all_ablation_results = []

n_layers = len(mha_s0_sft_model.layers)
n_heads = mha_s0_sft_info["config"]["n_heads"]

for layer in range(n_layers):
    print(f"Running layer {layer}/{n_layers-1}...")
    
    for head in range(n_heads):
        
        with mean_ablate_heads(
            mha_s0_sft_model,
            {layer: [head]},
            mha_s0_sft_head_means
        ):
            acc, margin, _ = get_ioi_metrics(
                mha_s0_sft_model,
                FIXED_IOI_500
            )
        
        all_ablation_results.append({
            "layer": layer,
            "head": head,
            "accuracy": acc,
            "mean_margin": margin,
            "accuracy_drop_pp": (
                mha_s0_sft_baseline_acc - acc
            ) * 100,
            "margin_drop": (
                mha_s0_sft_baseline_margin - margin
            )
        })

print("Done!")
print("Total results:", len(all_ablation_results))

In [ ]:
import pandas as pd

mha_s0_sft_ablation_df = pd.DataFrame(all_ablation_results)

mha_s0_sft_ablation_df.to_csv(
    "/kaggle/working/mha_s0_sft_ablation_results.csv",
    index=False
)

print("Saved successfully!")

In [ ]:
import os
print(os.path.exists("YOUR_SAVED_FILE_PATH"))

In [ ]:
import os

print("Current working directory:", os.getcwd())
print("\nFiles in /kaggle/working:")
print(os.listdir("/kaggle/working"))

In [ ]:
# Search for CSV and pickle/result files in the working directory
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith((".csv", ".pkl", ".pt", ".json")):
            print(os.path.join(root, file))

In [ ]:
import shutil
import os

files_to_save = [
    "/kaggle/working/key_ablation_results.csv",
    "/kaggle/working/mha_s0_sft_ablation_results.csv",
    "/kaggle/working/mha_vs_gqa_comparison.csv",
    "/kaggle/working/fixed_ioi_500.json",
    "/kaggle/working/gqa_l10h4_verification.csv",
    "/kaggle/working/l6h8_verification.csv",
]

# Create a folder containing only your important results
backup_dir = "/kaggle/working/research_backup"
os.makedirs(backup_dir, exist_ok=True)

for file_path in files_to_save:
    if os.path.exists(file_path):
        shutil.copy(file_path, backup_dir)

# Create ZIP
shutil.make_archive(
    "/kaggle/working/research_backup",
    "zip",
    backup_dir
)

print("Backup created!")
print("/kaggle/working/research_backup.zip")

In [ ]:
import os

zip_path = "/kaggle/working/research_backup.zip"

print("ZIP exists:", os.path.exists(zip_path))

if os.path.exists(zip_path):
    print("Size:", os.path.getsize(zip_path), "bytes")
    print("Files currently in working:")
    print(os.listdir("/kaggle/working"))

In [ ]:
import pandas as pd
import os
import torch

# All SFT checkpoint paths
SFT_CHECKPOINTS = {
    "mha_s0_sft": "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s0-datset/ckpts/mha_s0_sft/final.pt",
    "mha_s1_sft": "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s1-datset/ckpts/mha_s1_sft/final.pt",
    "mha_s2_sft": "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s2-datset/ckpts/mha_s2_sft/final.pt",
    "gqa_s0_sft": "/kaggle/input/datasets/nainikabatchu/sft-runner-gqa-s0-datset/ckpts/gqa_s0_sft/final.pt",
    "gqa_s1_sft": "/kaggle/input/datasets/nainikabatchu/sft-runner-gqa-s1-datset/ckpts/gqa_s1_sft/final.pt",
    "gqa_s2_sft": "/kaggle/input/datasets/nainikabatchu/sft-runner-gqa-s2-datset/ckpts/gqa_s2_sft/final.pt",
}

save_path = "/kaggle/working/sft_baseline_results.csv"
summary = []

for name, path in SFT_CHECKPOINTS.items():
    print(f"\n{'='*50}")
    print(f"Evaluating {name}...")

    # Load model using your existing function
    model, info = load_model(path, device=device)

    # Evaluate on the same 500 IOI examples
    accuracy, mean_margin, _ = get_ioi_metrics(
        model,
        FIXED_IOI_500
    )

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Mean margin: {mean_margin:.4f}")

    summary.append({
        "model": name,
        "accuracy": accuracy,
        "mean_margin": mean_margin,
        "step": info.get("step"),
        "tokens_seen": info.get("tokens_seen")
    })

    # SAVE AFTER EVERY MODEL
    pd.DataFrame(summary).to_csv(save_path, index=False)
    print(f"Saved progress → {save_path}")

    # Free GPU memory before the next model
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nDONE — all completed results:")
display(pd.DataFrame(summary))

In [ ]:
import os

print(os.listdir("/kaggle/working"))

In [ ]:
from IPython.display import FileLink, display

files_to_download = [
    "/kaggle/working/research_backup.zip",
    "/kaggle/working/sft_baseline_results.csv"
]

for file in files_to_download:
    display(FileLink(file))

In [ ]:
import os
import pandas as pd
import torch

# ============================================================
# TARGETED MHA S1 SFT ABLATION
# Tests all heads in the most relevant layers: 6, 7, and 11
# ============================================================

MODEL_NAME = "mha_s1_sft"
OUTPUT_FILE = f"/kaggle/working/{MODEL_NAME}_targeted_ablation.csv"

# Your completed baseline results
baseline_acc = 1.000
baseline_margin = 6.110499

# Layers selected based on previous MHA ablation findings
candidate_layers = [6, 7, 11]
n_heads = 12

# ------------------------------------------------------------
# 1. Compute reference mean activations
# ------------------------------------------------------------
print("Computing reference mean activations...")

head_means = compute_reference_mean_activations(
    mha_s1_sft_model,
    FIXED_IOI_500
)

print("Reference means ready.")

# ------------------------------------------------------------
# 2. Resume from previous progress if the file already exists
# ------------------------------------------------------------
if os.path.exists(OUTPUT_FILE):
    results = pd.read_csv(OUTPUT_FILE).to_dict("records")
    completed = {
        (int(r["layer"]), int(r["head"]))
        for r in results
    }
    print(f"Resuming: {len(completed)} heads already completed.")
else:
    results = []
    completed = set()

# ------------------------------------------------------------
# 3. Run targeted ablations
# ------------------------------------------------------------
total = len(candidate_layers) * n_heads
done_count = len(completed)

for layer in candidate_layers:
    for head in range(n_heads):

        if (layer, head) in completed:
            print(f"Skipping Layer {layer}, Head {head} (already done)")
            continue

        print(
            f"\n[{done_count + 1}/{total}] "
            f"Running Layer {layer}, Head {head}..."
        )

        with mean_ablate_heads(
            mha_s1_sft_model,
            {layer: [head]},
            head_means
        ):
            acc, margin, _ = get_ioi_metrics(
                mha_s1_sft_model,
                FIXED_IOI_500
            )

        result = {
            "model": MODEL_NAME,
            "layer": layer,
            "head": head,
            "accuracy": acc,
            "mean_margin": margin,
            "accuracy_drop_pp": (baseline_acc - acc) * 100,
            "margin_drop": baseline_margin - margin
        }

        results.append(result)
        done_count += 1

        # SAVE AFTER EVERY HEAD
        pd.DataFrame(results).to_csv(
            OUTPUT_FILE,
            index=False
        )

        print(
            f"Done | Accuracy: {acc:.4f} | "
            f"Margin: {margin:.4f} | "
            f"Margin drop: {baseline_margin - margin:.4f}"
        )
        print(f"Progress saved → {OUTPUT_FILE}")

# ------------------------------------------------------------
# 4. Show most important heads found
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("TARGETED ABLATION COMPLETE")
print("=" * 60)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    "margin_drop",
    ascending=False
)

display(results_df)

print("\nTop 10 most important heads by margin drop:")
display(results_df.head(10))

In [ ]:
[name for name in globals().copy()
 if "model" in name.lower()]


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "sft" in root.lower() and "ckpt" in root.lower():
        print(root)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/nainikabatchu"):
    if root.endswith("_sft"):
        print("\nMODEL FOLDER:", root)
        print("FILES:", files)

In [ ]:
print("Transformer:", "Transformer" in globals())
print("Tokenizer:", "tokenizer" in globals())
print("Dataset:", "FIXED_IOI_500" in globals())
print("Metrics:", "get_ioi_metrics" in globals())
print("Mean ablation:", "mean_ablate_heads" in globals())
print("Device:", device)

In [ ]:
MHA_S1_SFT_PATH = "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s1-datset/ckpts/mha_s1_sft/final.pt"

In [ ]:
import torch

path = "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s1-datset/ckpts/mha_s1_sft/final.pt"

checkpoint = torch.load(path, map_location=device)

print(type(checkpoint))

if isinstance(checkpoint, dict):
    print(checkpoint.keys())
else:
    print(checkpoint)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [ ]:
path = "/kaggle/input/datasets/nainikabatchu/sft-runner-mha-s1-datset/ckpts/mha_s1_sft/final.pt"

checkpoint = torch.load(path, map_location=device)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Keys:", checkpoint.keys())
else:
    print(checkpoint)

In [ ]:
print("CONFIG:")
print(checkpoint["config"])

print("\nMODEL TYPE:")
print(type(checkpoint["model"]))

print("\nFIRST FEW MODEL KEYS:")
print(list(checkpoint["model"].keys())[:10])

print("\nSTEP:", checkpoint["step"])
print("TOKENS SEEN:", checkpoint["tokens_seen"])

In [ ]:
import os
import sys

CODE_DIR = "/kaggle/input/datasets/nainikabatchu/archive"

print(os.listdir(CODE_DIR))

sys.path.append(CODE_DIR)

In [ ]:
from model import Transformer
from checkpoint import load_model

print("Transformer loaded:", Transformer)
print("load_model loaded:", load_model)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

In [ ]:
MHA_S1_PATH = (
    "/kaggle/input/datasets/nainikabatchu/"
    "sft-runner-mha-s1-datset/ckpts/mha_s1_sft/final.pt"
)

mha_s1_sft_model, mha_s1_sft_config = load_model(
    MHA_S1_PATH,
    device=device
)

mha_s1_sft_model.eval()

print("MHA-S1-SFT loaded successfully")
print(mha_s1_sft_config)

In [ ]:
mha_s1_acc, mha_s1_margin, _ = get_ioi_metrics(
    mha_s1_sft_model,
    FIXED_IOI_500
)

print(f"Accuracy: {mha_s1_acc:.4f}")
print(f"Mean margin: {mha_s1_margin:.4f}")

In [ ]:
def get_ioi_metrics(model, test_set):
    """Compute IOI accuracy and mean logit margin."""

    results = []
    model.eval()

    for ex in test_set:
        correct_id = get_single_token_id(ex["correct"])
        incorrect_id = get_single_token_id(ex["incorrect"])

        ids = tokenizer.encode(ex["prompt"]).ids
        input_ids = torch.tensor(
            [ids],
            dtype=torch.long,
            device=device
        )

        with torch.no_grad():
            batch_logits, _ = model(input_ids)

        next_logits = batch_logits[0, -1]

        correct_score = next_logits[correct_id].item()
        incorrect_score = next_logits[incorrect_id].item()
        margin = correct_score - incorrect_score

        results.append({
            "success": margin > 0,
            "margin": margin
        })

    accuracy = sum(r["success"] for r in results) / len(results)
    mean_margin = sum(r["margin"] for r in results) / len(results)

    return accuracy, mean_margin, results

In [ ]:
print("tokenizer:", "tokenizer" in globals())
print("get_single_token_id:", "get_single_token_id" in globals())
print("FIXED_IOI_500:", "FIXED_IOI_500" in globals())
print("mha_s1_sft_model:", "mha_s1_sft_model" in globals())

In [ ]:
import os
import json
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Model exists:", "mha_s1_sft_model" in globals())

In [ ]:
with open("/kaggle/working/fixed_ioi_500.json", "r") as f:
    FIXED_IOI_500 = json.load(f)

print("Loaded examples:", len(FIXED_IOI_500))
print("First example:", FIXED_IOI_500[0])

In [ ]:
import os

print("GPU available:")
import torch
print(torch.cuda.is_available())

print("\nFine-tuned checkpoints:")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "final.pt" and "sft" in root.lower():
            print(os.path.join(root, file))

In [ ]:

import os
import torch
import pandas as pd

SFT_CHECKPOINTS = {
    name: (
        f"/kaggle/input/datasets/nainikabatchu/"
        f"sft-runner-{name.replace('_', '-')}-datset/"
        f"ckpts/{name}_sft/final.pt"
    )
    for name in [
        "mha_s0", "mha_s1", "mha_s2",
        "gqa_s0", "gqa_s1", "gqa_s2"
    ]
}

rows = []

for name, path in SFT_CHECKPOINTS.items():
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )
    cfg = ckpt["config"]

    rows.append({
        "model": name,
        "layers": cfg.get("n_layers"),
        "heads": cfg.get("n_heads"),
        "kv_heads": cfg.get("n_kv_heads"),
        "step": ckpt.get("step"),
        "has_weights": "model" in ckpt
    })

    del ckpt

summary = pd.DataFrame(rows)
display(summary)

print("All checkpoints inspected successfully!")


In [ ]:

required = [
    "load_model",
    "tokenizer",
    "FIXED_IOI_500",
    "get_ioi_metrics",
    "mean_ablate_heads",
    "compute_reference_mean_activations"
]

missing = [name for name in required if name not in globals()]

if missing:
    print("Missing:", missing)
    print("Restore these from the original notebook first.")
else:
    print("All required functions and data are ready!")
    print("Evaluation examples:", len(FIXED_IOI_500))


In [ ]:

import os
import json
import zipfile

matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename == "fixed_ioi_500.json":
            matches.append(("json", os.path.join(root, filename)))
        elif filename == "research_backup.zip":
            matches.append(("zip", os.path.join(root, filename)))

if not matches:
    print("Upload research_backup.zip as a Kaggle Dataset first.")
else:
    kind, path = matches[0]

    if kind == "json":
        with open(path) as f:
            FIXED_IOI_500 = json.load(f)
    else:
        with zipfile.ZipFile(path) as z:
            FIXED_IOI_500 = json.loads(
                z.read("fixed_ioi_500.json")
            )

    print("Restored prompts:", len(FIXED_IOI_500))
    print("Example:", FIXED_IOI_500[0])


In [ ]:

import json
import os

notebook_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename.startswith("part4-circuit-audit") and filename.endswith(".ipynb"):
            notebook_path = os.path.join(root, filename)
            break
    if notebook_path:
        break

if notebook_path is None:
    print("Attach your original .ipynb as a Kaggle Dataset first.")
else:
    with open(notebook_path, encoding="utf-8") as f:
        notebook = json.load(f)

    # Original notebook cells, zero-indexed
    restore = {
        22: "get_ioi_metrics",
        76: "mean_ablate_heads",
        140: "compute_reference_mean_activations"
    }

    for cell_index, function_name in restore.items():
        source = "".join(
            notebook["cells"][cell_index]["source"]
        )

        # Only execute the function definition, not later
        # experimental calls in the same cell.
        import ast
        tree = ast.parse(source)

        selected = [
            node for node in tree.body
            if isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            ) and node.name == function_name
        ]

        if not selected:
            raise ValueError(f"Could not find {function_name}")

        # Keep required imports and decorators
        imports = [
            node for node in tree.body
            if isinstance(node, (ast.Import, ast.ImportFrom))
        ]

        module = ast.Module(
            body=imports + selected,
            type_ignores=[]
        )
        exec(compile(module, notebook_path, "exec"), globals())

        print("Restored:", function_name)


In [ ]:

import torch
import os
import time

print("GPU:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

checkpoints = []

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename == "final.pt":
            checkpoints.append(os.path.join(root, filename))

print("Checkpoints found:", len(checkpoints))

for path in checkpoints:
    print(path)

print("Current session ready!")


In [ ]:

required = [
    "ModelConfig",
    "Transformer",
    "tokenizer",
    "get_single_token_id",
    "load_model_from_checkpoint",
    "FIXED_IOI_500",
    "get_ioi_metrics",
    "mean_ablate_heads",
    "compute_reference_mean_activations"
]

missing = []

for name in required:
    if name in globals():
        print(f"READY   : {name}")
    else:
        print(f"MISSING : {name}")
        missing.append(name)

print("\nTotal missing:", len(missing))


In [ ]:

import inspect

functions = [
    "load_model_from_checkpoint",
    "get_ioi_metrics",
    "mean_ablate_heads",
    "compute_reference_mean_activations"
]

for name in functions:
    if name in globals():
        print(name, inspect.signature(globals()[name]))
    else:
        print(name, "MISSING")


In [ ]:

import os
import torch
import pandas as pd

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = "/kaggle/working/sft_audit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GQA_S0_PATH = (
    "/kaggle/input/datasets/nainikabatchu/"
    "sft-runner-gqa-s0-datset/"
    "ckpts/gqa_s0_sft/final.pt"
)

# Load only GQA S0
model, checkpoint = load_model_from_checkpoint(
    GQA_S0_PATH
)

model.eval()

# Evaluate on the same 500 IOI prompts
baseline_acc, baseline_margin, _ = get_ioi_metrics(
    model,
    FIXED_IOI_500
)

print("\nGQA S0 BASELINE RESULTS")
print("Accuracy:", baseline_acc)
print("Logit margin:", baseline_margin)

# Save baseline immediately
pd.DataFrame([{
    "model": "gqa_s0",
    "accuracy": baseline_acc,
    "logit_margin": baseline_margin
}]).to_csv(
    f"{OUTPUT_DIR}/gqa_s0_baseline.csv",
    index=False
)

print("Baseline saved successfully!")


In [ ]:

import time
import torch

start = time.time()

head_means = compute_reference_mean_activations(
    model,
    FIXED_IOI_500
)

print("Time taken:", round(time.time() - start, 2), "seconds")
print("Reference activation type:", type(head_means))
print("Reference activations ready!")


In [ ]:

import time
import inspect

print("Ablation function:")
print(inspect.getsource(mean_ablate_heads))


In [ ]:

import os
import time
import pandas as pd
import torch

output_file = "/kaggle/working/sft_audit/gqa_s0_head_ablation.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Benchmark one head
start = time.time()

with mean_ablate_heads(model, {0: [0]}, head_means):
    test_acc, test_margin, _ = get_ioi_metrics(
        model, FIXED_IOI_500
    )

elapsed = time.time() - start

print(f"One head took: {elapsed:.2f} seconds")
print(f"Estimated 144 heads: {elapsed * 144 / 60:.1f} minutes")

# Load previously saved results
if os.path.exists(output_file):
    results = pd.read_csv(output_file).to_dict("records")
else:
    results = []

completed = {
    (int(r["layer"]), int(r["head"]))
    for r in results
}

# Run only if estimated time is reasonable
if elapsed * 144 <= 20 * 60:

    for layer in range(12):
        for head in range(12):

            if (layer, head) in completed:
                continue

            with mean_ablate_heads(
                model, {layer: [head]}, head_means
            ):
                acc, margin, _ = get_ioi_metrics(
                    model, FIXED_IOI_500
                )

            results.append({
                "model": "gqa_s0",
                "layer": layer,
                "head": head,
                "accuracy": acc,
                "logit_margin": margin,
                "accuracy_drop": baseline_acc - acc,
                "margin_drop": baseline_margin - margin
            })

            # Save immediately
            pd.DataFrame(results).to_csv(
                output_file, index=False
            )

        print(f"Layer {layer + 1}/12 completed")

    print("GQA S0 HEAD ABLATION COMPLETE!")

else:
    print("Estimated runtime exceeds 20 minutes.")
    print("Full ablation skipped to protect our time budget.")


In [ ]:

import os
import gc
import time
import torch
import pandas as pd

OUTPUT_DIR = "/kaggle/working/sft_audit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

remaining_models = ["mha_s1", "gqa_s1", "mha_s2", "gqa_s2"]

for model_name in remaining_models:
    path = SFT_CHECKPOINTS[model_name]
    output_file = f"{OUTPUT_DIR}/{model_name}_head_ablation.csv"
    baseline_file = f"{OUTPUT_DIR}/{model_name}_baseline.csv"

    if os.path.exists(output_file):
        results = pd.read_csv(output_file).to_dict("records")
    else:
        results = []

    completed = {
        (int(r["layer"]), int(r["head"]))
        for r in results
    }

    if len(completed) == 144:
        print(f"{model_name}: already complete, skipping")
        continue

    # Free previous model memory
    if "model" in globals():
        del model
    if "head_means" in globals():
        del head_means
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\nStarting {model_name}")

    model, checkpoint = load_model_from_checkpoint(path)
    model.eval()

    baseline_acc, baseline_margin, _ = get_ioi_metrics(
        model, FIXED_IOI_500
    )

    pd.DataFrame([{
        "model": model_name,
        "accuracy": baseline_acc,
        "logit_margin": baseline_margin
    }]).to_csv(baseline_file, index=False)

    head_means = compute_reference_mean_activations(
        model, FIXED_IOI_500
    )

    for layer in range(12):
        for head in range(12):
            if (layer, head) in completed:
                continue

            with mean_ablate_heads(
                model, {layer: [head]}, head_means
            ):
                acc, margin, _ = get_ioi_metrics(
                    model, FIXED_IOI_500
                )

            results.append({
                "model": model_name,
                "layer": layer,
                "head": head,
                "accuracy": acc,
                "logit_margin": margin,
                "accuracy_drop": baseline_acc - acc,
                "margin_drop": baseline_margin - margin
            })

            pd.DataFrame(results).to_csv(
                output_file, index=False
            )

        print(f"{model_name}: layer {layer + 1}/12 complete")

    print(f"{model_name}: COMPLETE")

print("\nALL REMAINING MODELS PROCESSED!")


In [ ]:

import os

SFT_CHECKPOINTS = {}

for root, dirs, files in os.walk("/kaggle/input"):
    if "final.pt" not in files:
        continue

    for name in [
        "mha_s0", "mha_s1", "mha_s2",
        "gqa_s0", "gqa_s1", "gqa_s2"
    ]:
        if os.path.basename(root) == f"{name}_sft":
            SFT_CHECKPOINTS[name] = os.path.join(
                root, "final.pt"
            )

print("Fine-tuned checkpoints found:")

for name, path in sorted(SFT_CHECKPOINTS.items()):
    print(name, "READY")

required = ["mha_s1", "gqa_s1", "mha_s2", "gqa_s2"]

assert all(name in SFT_CHECKPOINTS for name in required), \
    "Some fine-tuned checkpoints are missing!"

print("\nAll four remaining checkpoints ready!")


In [ ]:

import os
import pandas as pd

folder = "/kaggle/working/sft_audit"

for name in [
    "mha_s0", "mha_s1", "mha_s2",
    "gqa_s0", "gqa_s1", "gqa_s2"
]:
    path = f"{folder}/{name}_head_ablation.csv"

    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"{name}: {len(df)} heads")
    else:
        print(f"{name}: NOT FOUND")


In [ ]:

import os
import zipfile
import pandas as pd
import io

OUTPUT_DIR = "/kaggle/working/sft_audit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

target = f"{OUTPUT_DIR}/mha_s0_head_ablation.csv"
found = False

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        path = os.path.join(root, filename)

        if filename == "mha_s0_sft_ablation_results.csv":
            df = pd.read_csv(path)
            found = True
            break

        if filename == "research_backup.zip":
            with zipfile.ZipFile(path) as z:
                matches = [
                    name for name in z.namelist()
                    if name.endswith(
                        "mha_s0_sft_ablation_results.csv"
                    )
                ]

                if matches:
                    df = pd.read_csv(
                        io.BytesIO(z.read(matches[0]))
                    )
                    found = True
                    break

    if found:
        break

if found:
    print("Original columns:", df.columns.tolist())
    print("Rows:", len(df))
    display(df.head())
else:
    print("MHA S0 backup not attached to Kaggle.")



In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUT = "/kaggle/working/sft_audit"
os.makedirs(OUT, exist_ok=True)

# Standardize the recovered MHA S0 results
mha0 = df.copy()

mha0 = mha0.rename(columns={
    "mean_margin": "logit_margin",
    "accuracy_drop_pp": "accuracy_drop"
})

# Convert percentage points to fraction
mha0["accuracy_drop"] = mha0["accuracy_drop"] / 100
mha0["model"] = "mha_s0"

mha0.to_csv(
    f"{OUT}/mha_s0_head_ablation.csv",
    index=False
)

# Combine all six models
models = [
    "mha_s0", "mha_s1", "mha_s2",
    "gqa_s0", "gqa_s1", "gqa_s2"
]

frames = []

for name in models:
    data = pd.read_csv(
        f"{OUT}/{name}_head_ablation.csv"
    )

    assert len(data) == 144, f"Incomplete: {name}"
    frames.append(data)

combined = pd.concat(frames, ignore_index=True)

combined["architecture"] = combined["model"].str[:3].str.upper()
combined["seed"] = combined["model"].str[-1].astype(int)

combined.to_csv(
    f"{OUT}/all_sft_head_ablations.csv",
    index=False
)

print("Total head-ablation results:", len(combined))

# Summary by model
summary = combined.groupby(
    ["architecture", "model"]
).agg(
    mean_accuracy_drop=("accuracy_drop", "mean"),
    max_accuracy_drop=("accuracy_drop", "max"),
    mean_margin_drop=("margin_drop", "mean"),
    max_margin_drop=("margin_drop", "max")
).reset_index()

summary.to_csv(
    f"{OUT}/sft_model_summary.csv",
    index=False
)

display(summary)

# Graph 1: Mean accuracy drop
plt.figure(figsize=(9, 5))
sns.barplot(
    data=summary,
    x="model",
    y="mean_accuracy_drop",
    hue="architecture"
)
plt.title("SFT: Mean Accuracy Drop After Head Ablation")
plt.ylabel("Accuracy Drop (fraction)")
plt.tight_layout()
plt.savefig(f"{OUT}/sft_accuracy_comparison.png", dpi=200)
plt.show()

# Graph 2: Mean logit-margin drop
plt.figure(figsize=(9, 5))
sns.barplot(
    data=summary,
    x="model",
    y="mean_margin_drop",
    hue="architecture"
)
plt.title("SFT: Mean Logit Margin Drop After Head Ablation")
plt.tight_layout()
plt.savefig(f"{OUT}/sft_margin_comparison.png", dpi=200)
plt.show()

# Graph 3: Six heatmaps
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, name in zip(axes.flat, models):
    data = combined[combined["model"] == name]
    heatmap = data.pivot(
        index="layer",
        columns="head",
        values="margin_drop"
    )

    sns.heatmap(
        heatmap,
        ax=ax,
        cmap="coolwarm",
        center=0
    )

    ax.set_title(name.upper())

plt.suptitle("SFT Head Importance: Logit Margin Drop")
plt.tight_layout()
plt.savefig(f"{OUT}/sft_head_heatmaps.png", dpi=200)
plt.show()

print("All SFT comparison files and graphs saved!")


In [ ]:

import shutil
import os

archive = shutil.make_archive(
    "/kaggle/working/SFT_Circuit_Audit_Results",
    "zip",
    "/kaggle/working/sft_audit"
)

print("Export ready:", archive)
print("Size:", round(os.path.getsize(archive) / 1024 / 1024, 2), "MB")
